# Decoupling Lateral Movement ($\Delta x$) and Material Change ($\phi_0$) in a Subwavelength Fluid Front

**Purpose.** The fluid-front study in `FluidFlow_Playground.ipynb` and Section 3 of
`TimeLapse_Processing.ipynb` showed that the 2D phase-plane fit can recover the lateral
advance $\Delta x$ of a sub-wavelength wetting front from a fixed baseline/monitor crop,
using a centroid-doubling correction. That pipeline works on **real-valued** migrated
images, which (as shown below) makes it mathematically impossible for it to ever report a
non-zero intercept. This notebook adds the complementary half of the story: a second,
**analytic-signal** estimator that genuinely frees the intercept $\phi_0$, so that for the
same family of scenarios we can show the two channels — geometric slope and material-change
intercept — read out independently and do not contaminate each other.

## 1. Theory recap

### 1.1 The phase plane

For a migrated baseline image $b(z,x)$ and monitor image $m(z,x)$, the 2D Fourier shift
theorem gives the cross-spectrum

$$\Xi(\kappa_z,\kappa_x) = B(\kappa_z,\kappa_x)\,M^{*}(\kappa_z,\kappa_x),$$

whose phase is, for a baseline/monitor pair related by a rigid translation $(\Delta z,\Delta x)$,
exactly a plane through the origin,

$$\Phi(\kappa_z,\kappa_x) = \kappa_z\,\Delta z + \kappa_x\,\Delta x .$$

A weighted least-squares (WLS) fit of this plane, discretised over $N$ frequency bins inside
a coherent passband and amplitude-weighted by $|\Xi_i|$,

$$\Phi_i = \kappa_{z,i}\,\Delta z + \kappa_{x,i}\,\Delta x + c ,\qquad i = 1,\dots,N,$$

deliberately includes a third, constant design column. \cref{sec:th-material-change} of the
thesis shows *why*: a sub-wavelength fracture that changes fill (air $\to$ water) at fixed
geometry produces a frequency-independent phase rotation $\Delta\theta$ — the *same* number
at every $(\kappa_z,\kappa_x)$ — because the geometric factor $d$ and the $-j\omega$
derivative factor of the thin-layer reflection coefficient cancel exactly in the
baseline/monitor ratio. Because the $\kappa_z$ and $\kappa_x$ design columns are zero-mean
over a symmetric passband, no slope can fit a perfectly flat target without increasing the
residual — so the optimum slopes go to zero and the constant column absorbs $\Delta\theta$
with zero residual: $c \equiv \phi_0 = \Delta\theta$. A real displacement and a real material
change littered through the *same* fit therefore land in orthogonal, non-contaminating
parameters: $(\Delta z,\Delta x)$ for movement, $\phi_0$ for material identity.

> **Caveat — what $\phi_0$ physically is, and a confound to keep in mind.** $\phi_0$ here is the
> same quantity GPR practitioners call *polarity reversal*: a 180° phase flip diagnostic of
> $\varepsilon_{r,\text{base}}<\varepsilon_{r,\text{mon}}$ vs. the reverse [Campo, "On GPR signal
> polarity reversal," IWAGPR 2021]. That paper's normal-incidence reflection coefficient,
> $\Gamma=(\sqrt{\varepsilon_{r1}}-\sqrt{\varepsilon_{r2}})/(\sqrt{\varepsilon_{r1}}+\sqrt{\varepsilon_{r2}})$,
> is algebraically identical to the `refl(n1, n2) = (n1-n2)/(n1+n2)` thin-layer model already
> used in `PhasePlaneFit.ipynb`'s Material Change Test ($n=\sqrt{\varepsilon_r}$) — independent
> corroboration that $\phi_0$ is physically well-posed for our ice ($\varepsilon_r\approx3.15$)
> $\to$ water ($\varepsilon_r=81$) contrast. The same paper, however, shows that a *non-planar* or
> multi-reflector target can corrupt a clean polarity signature through internal wave
> interaction (e.g. creeping waves "erasing" a circular void's expected reversal) even with no
> attenuation at all — and our fluid front is a nine-box graded ramp, not a single flat
> interface. That mechanism is flagged again, with a concrete number attached, in \S4.3.

### 1.2 Why the existing fluid-front pipeline cannot see $\phi_0$

Section 3 of `TimeLapse_Processing.ipynb` fits this plane directly on the **real-valued**
migrated images. For a real-valued image, $B(-\kappa)=B^{*}(\kappa)$, so the cross-spectrum is
Hermitian-symmetric, $\Xi(-\kappa)=\Xi^{*}(\kappa)$, which forces $\Phi(-\kappa)=-\Phi(\kappa)$:
an *odd* function of $\kappa$. Combined with a passband mask and weights that are themselves
even in $\kappa$ ($|\Xi(-\kappa)|=|\Xi(\kappa)|$), every $+\kappa$ observation is exactly
cancelled by its $-\kappa$ mirror in the WLS normal equations — the fitted intercept is
**identically zero by symmetry**, regardless of how much material actually changed. All of the
front's information is consequently forced entirely into the slope term, which is why Section 3
reads $\Delta x$ off a *kx*-slope and applies the empirical centroid-doubling correction
$\Delta x_\text{true} = 2\,\Delta x_\text{raw}$ (the newly-wetted region spans
$[x_0, x_0+\Delta x_\text{true}]$, so its energy centroid sits at $x_0+\Delta x_\text{true}/2$).
That estimate is a real, useful number — but it is not a genuine test of the
geometry/material-change decoupling, because $\phi_0$ was never allowed to be anything but zero.

### 1.3 Unlocking $\phi_0$: the analytic-signal, $\kappa_z>0$ channel

To let $\phi_0$ move, the Hermitian symmetry must be broken **before** the 2D FFT. Taking the
analytic signal of each cropped image along $z$ (`scipy.signal.hilbert`, axis=0) suppresses
the negative-$\kappa_z$ half of the spectrum, leaving only the positive-$\kappa_z$ content that
actually carries the wavelet's energy. The fit is then additionally restricted to
$\kappa_z>0$ (`kz_pos_only=True`) — both to discard the (now near-empty) negative-$\kappa_z$
bins and, more importantly, to avoid the residual ill-conditioning a *one-sided* narrowband
spectrum otherwise creates between $\phi_0$ and $\kappa_z\Delta z$ (they become nearly
collinear once the negative mirror that disambiguates them is gone). Because the fracture's
vertical geometry truly does not move in this experiment, that last degree of freedom is fixed
at zero explicitly (`force_dz_zero=True`), leaving a clean 2-parameter fit for
$(\Delta x, \phi_0)$ on a well-conditioned design matrix.

### 1.4 What this notebook tests

For each lateral-advance scenario ($2\lambda, 1\lambda, \tfrac12\lambda, \tfrac14\lambda,
\tfrac18\lambda$) and each migration method, the **same fixed crop** (anchored to the baseline
front position, never re-centred on the monitor) is fed through both channels:

| Channel | Input | Free parameters | Reads out |
|---|---|---|---|
| Real / Hermitian | real $b,m$ | $(\Delta z,\Delta x)$, $\phi_0\equiv0$ by symmetry | $\Delta x_\text{inferred}=2\,\Delta x_\text{raw}$ — the front's advance |
| Analytic / $\kappa_z>0$ | Hilbert($b$), Hilbert($m$) along $z$ | $\Delta x$ ($\Delta z\equiv0$ fixed) and $\phi_0$ | $\phi_0$ — the material-change signature; $\Delta x$ here is a *residual* that should stay near zero |

If the decoupling argument of \cref{sec:th-material-change} holds for a real, distributed
target (not just an idealised point scatterer), $\phi_0$ should stay **stable across all five
scales** — the same air$\to$water contrast every time — while the residual slope in the
analytic channel stays near zero, confirming the front's *advance* is being read correctly by
the real channel and not leaking into the material-change channel.

## 2. Setup & Imports

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.signal         import hilbert as sp_hilbert
from scipy.signal.windows import tukey

plt.rcParams['figure.dpi'] = 110


In [ ]:
# ── Ice physics & migration grid (identical to TimeLapse_Processing.ipynb / FluidFlow_Playground.ipynb) ──
v_ice = 0.168    # m/ns, exploding-reflector full medium velocity
f_c   = 1.5      # GHz, Ricker centre frequency
lam   = v_ice / f_c   # dominant wavelength, ~112.5 mm

# ── Lateral-advance scenarios under test (subset of the 7 simulated in FluidFlow_Playground.ipynb) ──
SCENARIOS_TO_USE = ['2λ', '1λ', '½λ', '¼λ', '⅛λ']

# ── Load the migrated fluid-front dataset ─────────────────────────────────────
FF_ROOT = Path(r'C:\Users\Administrator\OneDrive\Thesis\TimeLapse_Notebooks\fluidflow_study')
ff = np.load(str(FF_ROOT / 'migrated_results.npz'), allow_pickle=False)

ff_kirchhoff = ff['kirchhoff']            # (n_scen, n_z, n_x)
ff_gazdag    = ff['gazdag']
ff_backprop  = ff['backprop']
ff_scenarios = [str(s) for s in ff['scenarios']]      # ['Baseline', '2λ', '1λ', ...]
ff_x_traces  = ff['x_traces']
ff_z_img     = ff['z_img']
ff_x_front   = ff['x_scatterer']          # true front position per scenario [m] (ground truth)

ff_dz_mig = float(ff_z_img[1]    - ff_z_img[0])
ff_dx_mig = float(ff_x_traces[1] - ff_x_traces[0])
ff_kz_c   = 2 * np.pi / lam

# True lateral advance of each scenario relative to the baseline front position
true_dx_by_scenario = {
    label: float(ff_x_front[i] - ff_x_front[0])
    for i, label in enumerate(ff_scenarios)
}

MIGRATION_METHODS = {
    'Kirchhoff': ff_kirchhoff,
    'Gazdag':    ff_gazdag,
    'Back-prop': ff_backprop,
}

print(f"lambda = {lam*1e3:.1f} mm   kz_c = {ff_kz_c:.1f} rad/m")
print(f"Grid:  dz = {ff_dz_mig*1e3:.2f} mm   dx = {ff_dx_mig*1e3:.2f} mm")
print("True Delta_x per scenario (mm):")
for label in SCENARIOS_TO_USE:
    print(f"  {label:>4s} : {true_dx_by_scenario[label]*1e3:6.2f} mm")


## 3. Core Processing Engine

`estimate_shift_2d` is reused verbatim from `TimeLapse_Processing.ipynb` — it is the single
function behind every phase-plane figure in the thesis. The two flags exercised here,
`kz_pos_only` and `force_dz_zero`, were added precisely to support the analytic-signal,
material-change channel described above.

In [ ]:
def estimate_shift_2d(base, mon, dz_g, dx_g, kz_cent, kz_pos_only=False, force_dz_zero=False):
    """
    Fit a 2D phase plane phi(kz,kx) = kz*dz + kx*dx + phi_0 to the cross-spectrum.
    Returns (dz_est, dx_est, phi_0, XS, kz_ax, kx_ax).

    kz_pos_only=True   -> restrict WLS to KZ > 0 (use with analytic-signal inputs).
    force_dz_zero=True -> fit only (dx, phi_0); KZ column dropped and dz_est returned
                         as 0.0. Use when vertical movement is known to be absent:
                         removes ill-conditioning from narrowband wavelet where phi_0
                         and kz*dz are nearly indistinguishable on the one-sided spectrum.
    """
    Nz, Nx = base.shape

    kz_ax = np.fft.fftfreq(Nz, d=dz_g) * 2 * np.pi
    kx_ax = np.fft.fftfreq(Nx, d=dx_g) * 2 * np.pi
    KZ, KX = np.meshgrid(kz_ax, kx_ax, indexing='ij')

    taper = np.outer(tukey(Nz, alpha=0.15), tukey(Nx, alpha=0.15))

    base_fft = np.fft.fft2(base * taper)
    mon_fft  = np.fft.fft2(mon  * taper)
    XS       = base_fft * np.conj(mon_fft)

    w   = np.abs(XS)
    phi = np.angle(XS)

    band = (np.abs(KZ) < 1.4 * kz_cent) & (np.abs(KX) < 1.4 * kz_cent)
    mask = (w > 0.10 * w.max()) & band & ((np.abs(KZ) + np.abs(KX)) > 0)
    if kz_pos_only:
        mask &= (KZ > 0)

    W = w[mask]
    if force_dz_zero:
        # 2-parameter fit: phi = kx*dx + phi_0  (dz = 0 by physics)
        A = np.column_stack([KX[mask], np.ones(mask.sum())])
        c = np.linalg.lstsq(A * W[:, None], phi[mask] * W, rcond=None)[0]
        return 0.0, c[0], c[1], XS, kz_ax, kx_ax
    else:
        # 3-parameter fit: phi = kz*dz + kx*dx + phi_0
        A = np.column_stack([KZ[mask], KX[mask], np.ones(mask.sum())])
        c = np.linalg.lstsq(A * W[:, None], phi[mask] * W, rcond=None)[0]
        return c[0], c[1], c[2], XS, kz_ax, kx_ax


In [ ]:
def locate_baseline_front(base_stack):
    """
    Baseline fluid-front x-position, located from the SMALLEST-shift scenario's
    diff-envelope peak (index -1, robust against migration artefacts that can
    shift the global envelope argmax to the wrong lateral position) -- identical
    to the localisation already used in TimeLapse_Processing.ipynb Section 3.
    """
    from scipy.signal import hilbert as _hilbert
    base_img = np.nan_to_num(base_stack[0])
    env_base = np.abs(_hilbert(base_img, axis=0))
    env_ref  = np.abs(_hilbert(np.nan_to_num(base_stack[-1]), axis=0))
    diff_env = np.abs(env_ref - env_base)
    _, ix_ref = np.unravel_index(np.argmax(diff_env), diff_env.shape)
    return ff_x_traces[ix_ref]


def crop_indices(x_apex_base, half_width_lam=2.5):
    """Fixed +/- half_width_lam crop window anchored to the BASELINE front --
    never re-centred on the monitor, so a real front advance can only show up
    as a slope/intercept in the fit, not be hidden by re-tracking it."""
    hw = half_width_lam * lam
    ix_lo = np.searchsorted(ff_x_traces, x_apex_base - hw)
    ix_hi = np.searchsorted(ff_x_traces, x_apex_base + hw)
    return ix_lo, ix_hi


def estimate_movement_and_material(base_crop, mon_crop):
    """
    Run both channels on the same fixed crop:
      - real / Hermitian channel  -> centroid-corrected lateral advance
      - analytic / kz>0 channel   -> material-change intercept phi_0,
                                      with a residual slope as a contamination check
    """
    # --- Channel 1: real-valued fit (phi_0 forced to 0 by Hermitian symmetry) ---
    dz_r, dx_r, phi0_r, *_ = estimate_shift_2d(base_crop, mon_crop, ff_dz_mig, ff_dx_mig, ff_kz_c)
    dx_inferred = 2.0 * dx_r   # centroid -> leading-edge correction

    # --- Channel 2: analytic-signal fit (phi_0 unlocked, dz fixed at 0) ---
    a_base = sp_hilbert(base_crop, axis=0)
    a_mon  = sp_hilbert(mon_crop,  axis=0)
    _, dx_a, phi0_a, *_ = estimate_shift_2d(
        a_base, a_mon, ff_dz_mig, ff_dx_mig, ff_kz_c,
        kz_pos_only=True, force_dz_zero=True,
    )

    return dict(
        dz_real=dz_r, dx_raw=dx_r, dx_inferred=dx_inferred, phi0_real_deg=np.degrees(phi0_r),
        dx_residual=dx_a, phi0_material_deg=np.degrees(phi0_a),
    )


## 4. Execution Loop

For every migration method and every lateral-advance scenario in `SCENARIOS_TO_USE`, crop the
baseline/monitor pair around the (fixed) baseline front position and run both channels.

In [ ]:
rows = []

for method_name, base_stack in MIGRATION_METHODS.items():

    x_apex_base = locate_baseline_front(base_stack)
    ix_lo, ix_hi = crop_indices(x_apex_base, half_width_lam=2.5)

    base_img  = np.nan_to_num(base_stack[0])
    base_crop = base_img[:, ix_lo:ix_hi]

    for label in SCENARIOS_TO_USE:
        i_scen   = ff_scenarios.index(label)
        mon_img  = np.nan_to_num(base_stack[i_scen])
        mon_crop = mon_img[:, ix_lo:ix_hi]

        result = estimate_movement_and_material(base_crop, mon_crop)
        result.update(
            method=method_name,
            scenario=label,
            true_dx_mm=true_dx_by_scenario[label] * 1e3,
        )
        rows.append(result)

results_df = pd.DataFrame(rows)
results_df['dx_inferred_mm']   = results_df['dx_inferred']   * 1e3
results_df['dx_residual_mm']   = results_df['dx_residual']   * 1e3
results_df['inference_err_mm'] = results_df['dx_inferred_mm'] - results_df['true_dx_mm']

results_df[['method', 'scenario', 'true_dx_mm', 'dx_inferred_mm',
            'inference_err_mm', 'phi0_material_deg', 'dx_residual_mm']]


### 4.1 Per-scenario diagnostic (Gazdag, representative method)

A single-method diagnostic in the same style as the existing Section 3 figures, but now showing
*both* channels side by side: the real-valued cross-spectrum phase (forced flat, $\phi_0=0$)
next to the analytic-signal cross-spectrum phase (free $\phi_0$, the material-change
signature).

In [ ]:
diag_method = 'Gazdag'
base_stack  = MIGRATION_METHODS[diag_method]
x_apex_base = locate_baseline_front(base_stack)
ix_lo, ix_hi = crop_indices(x_apex_base, half_width_lam=2.5)
x_crop = ff_x_traces[ix_lo:ix_hi]

base_img  = np.nan_to_num(base_stack[0])
base_crop = base_img[:, ix_lo:ix_hi]

n_cases = len(SCENARIOS_TO_USE)
fig, axes = plt.subplots(n_cases, 3, figsize=(15, 3.3 * n_cases))
fig.suptitle(
    f"FluidFlow -- {diag_method} | Real channel (phi_0 = 0) vs Analytic channel (phi_0 free) | "
    f"crop +/-2.5lam @ baseline front",
    fontsize=11, fontweight='bold', y=1.01,
)

klim = 1.5 * ff_kz_c

for row, label in enumerate(SCENARIOS_TO_USE):
    i_scen   = ff_scenarios.index(label)
    mon_img  = np.nan_to_num(base_stack[i_scen])
    mon_crop = mon_img[:, ix_lo:ix_hi]

    dz_r, dx_r, phi0_r, XS_r, kz_ax, kx_ax = estimate_shift_2d(
        base_crop, mon_crop, ff_dz_mig, ff_dx_mig, ff_kz_c
    )
    a_base = sp_hilbert(base_crop, axis=0)
    a_mon  = sp_hilbert(mon_crop,  axis=0)
    _, dx_a, phi0_a, XS_a, *_ = estimate_shift_2d(
        a_base, a_mon, ff_dz_mig, ff_dx_mig, ff_kz_c,
        kz_pos_only=True, force_dz_zero=True,
    )

    true_dx_mm = true_dx_by_scenario[label] * 1e3

    def _phase_panel(ax, XS, title):
        XS_s   = np.fft.fftshift(XS)
        kz_s   = np.fft.fftshift(kz_ax)
        kx_s   = np.fft.fftshift(kx_ax)
        energy = np.abs(XS_s)
        phi_show = np.where(energy > 0.01 * energy.max(),
                             np.degrees(np.angle(XS_s)), np.nan)
        im = ax.pcolormesh(kx_s, kz_s, phi_show, cmap='RdBu_r',
                            vmin=-180, vmax=180, shading='auto')
        for sgn in [-1, 1]:
            ax.axhline(sgn * klim, color='k', lw=0.8, ls='--', alpha=0.4)
            ax.axvline(sgn * klim, color='k', lw=0.8, ls='--', alpha=0.4)
        ax.set_title(title, fontsize=9)
        ax.set_xlabel('kx [rad/m]'); ax.set_ylabel('kz [rad/m]')
        ax.set_xlim(-klim * 2.5, klim * 2.5); ax.set_ylim(-klim * 2.5, klim * 2.5)
        return im

    im0 = _phase_panel(
        axes[row, 0], XS_r,
        f"{label}  (true dx={true_dx_mm:.1f} mm)\nReal channel: phi_0 forced 0 -> all info in kx slope",
    )
    plt.colorbar(im0, ax=axes[row, 0], fraction=0.046)

    im1 = _phase_panel(
        axes[row, 1], XS_a,
        "Analytic channel ( kz>0 ): phi_0 free\n(material-change signature)",
    )
    plt.colorbar(im1, ax=axes[row, 1], fraction=0.046)

    ax2 = axes[row, 2]
    ax2.axis('off')
    txt = (
        f"True front advance:    {true_dx_mm:+7.2f} mm\n\n"
        f"-- Real channel --\n"
        f"  dx_raw (slope):       {dx_r*1e3:+7.3f} mm\n"
        f"  dx_inferred (2x):     {2*dx_r*1e3:+7.3f} mm\n"
        f"  phi_0 (forced):       {np.degrees(phi0_r):+7.2f} deg\n\n"
        f"-- Analytic channel --\n"
        f"  dx residual:          {dx_a*1e3:+7.3f} mm  (expect ~0)\n"
        f"  phi_0 (material):     {np.degrees(phi0_a):+7.2f} deg\n"
    )
    ax2.text(0.05, 0.95, txt, transform=ax2.transAxes, fontsize=10, va='top',
             family='monospace', bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.show()


### 4.2 Per-Method Calibration -- correcting the Back-propagation over-correction

Section 4 applied a single, universal $\times2$ centroid correction to every method, inherited
from Section 3 of `TimeLapse_Processing.ipynb`. The diagnostic above suggests this is not
warranted for Back-propagation, whose raw $kx$-slope already lands close to the *full*
$\Delta x_\text{true}$ rather than half of it. Instead of assuming a factor, derive it
empirically per method from the four scales where the fixed $\pm2.5\lambda$ crop is known to
behave ($1\lambda$ down to $\tfrac18\lambda$; $2\lambda$ is excluded here because its crop
margin is shown in \S4.3 to be too tight for *any* method, which would bias the calibration):

$$k_\text{method} = \mathrm{median}_i\left(\frac{\Delta x_{\text{true},i}}{\Delta x_{\text{raw},i}}\right).$$

In [ ]:
CALIBRATION_SCENARIOS = ['1λ', '½λ', '¼λ', '⅛λ']   # excludes 2λ -- see Section 4.3

calibration_factors = {}
for method in MIGRATION_METHODS:
    sub = results_df[(results_df['method'] == method) & (results_df['scenario'].isin(CALIBRATION_SCENARIOS))]
    k = float(np.median(sub['true_dx_mm'] / (sub['dx_raw'] * 1e3)))
    calibration_factors[method] = k

print("Empirical calibration factor k = median(true_dx / dx_raw):")
for m, k in calibration_factors.items():
    flag = '' if abs(k - 2.0) < 0.3 else '  <-- NOT the assumed centroid factor of 2'
    print(f"  {m:>10s}: k = {k:.3f}{flag}")

results_df['calib_factor']      = results_df['method'].map(calibration_factors)
results_df['dx_calibrated_mm']  = results_df['dx_raw'] * 1e3 * results_df['calib_factor']
results_df['calibrated_err_mm'] = results_df['dx_calibrated_mm'] - results_df['true_dx_mm']

results_df[['method', 'scenario', 'true_dx_mm', 'dx_inferred_mm',
            'dx_calibrated_mm', 'calibrated_err_mm']]


Kirchhoff ($k\approx1.97$) and Gazdag ($k\approx2.00$) confirm the theoretical centroid
factor of 2 directly from the data. Back-propagation comes out at $k\approx0.99$ -- i.e. its raw
slope already *is* the front displacement, with no centroid dilution -- so the calibrated
estimate now uses $\times1$ for that method instead of the blind $\times2$, removing the
roughly $2\times$ over-correction seen in Section 4.

### 4.3 Crop-Width Sensitivity -- isolating the $2\lambda$ artifact from genuine $\phi_0$ scale-dependence

Section 4's results showed two things that look superficially similar but need to be told apart:
the centroid estimate collapsing to exactly $0$ mm at $2\lambda$, and $\phi_0$ being noticeably
larger at $2\lambda$ than at the other four scales. Both could be the *same* fixed
$\pm2.5\lambda$-crop artifact (the new front position sits only $0.5\lambda$ inside the crop
edge at $2\lambda$), or $\phi_0$ could genuinely scale with $\Delta x_\text{true}$ regardless of
crop size. Re-running both channels at several crop half-widths distinguishes the two.

In [ ]:
CROP_WIDTHS_LAM = [2.5, 3.5, 4.5, 6.0]

crop_rows = []
for hw in CROP_WIDTHS_LAM:
    for method_name, base_stack in MIGRATION_METHODS.items():
        x_apex_base = locate_baseline_front(base_stack)
        ix_lo, ix_hi = crop_indices(x_apex_base, half_width_lam=hw)
        base_img  = np.nan_to_num(base_stack[0])
        base_crop = base_img[:, ix_lo:ix_hi]
        for label in SCENARIOS_TO_USE:
            i_scen   = ff_scenarios.index(label)
            mon_img  = np.nan_to_num(base_stack[i_scen])
            mon_crop = mon_img[:, ix_lo:ix_hi]
            res = estimate_movement_and_material(base_crop, mon_crop)
            crop_rows.append(dict(
                method=method_name, scenario=label, crop_hw_lam=hw,
                phi0_material_deg=res['phi0_material_deg'],
                dx_inferred_mm=res['dx_inferred'] * 1e3,
                true_dx_mm=true_dx_by_scenario[label] * 1e3,
            ))

crop_df = pd.DataFrame(crop_rows)

print("phi_0 [deg] vs crop half-width, by method/scenario:")
print(crop_df.pivot_table(index=['method', 'scenario'], columns='crop_hw_lam',
                           values='phi0_material_deg').loc[:, CROP_WIDTHS_LAM].round(3))

print("\ndx_inferred (blind x2) [mm] vs crop half-width -- watch the 2lam row recover:")
print(crop_df.pivot_table(index=['method', 'scenario'], columns='crop_hw_lam',
                           values='dx_inferred_mm').loc[:, CROP_WIDTHS_LAM].round(1))

# How strongly does phi_0 track the (supposedly unrelated) residual geometric slope,
# at the original +/-2.5lam crop used throughout Section 4? A high correlation means
# geometry and material change are NOT cleanly separated for that method.
print("\ncorr(phi0_material_deg, dx_residual_mm) across the 5 scenarios, per method (+/-2.5lam crop):")
for method in MIGRATION_METHODS:
    sub = results_df[results_df['method'] == method]
    rho = sub['phi0_material_deg'].corr(sub['dx_residual_mm'])
    print(f"  {method:>10s}: rho = {rho:+.3f}")


**Finding.** Widening the crop resolves the $2\lambda$ collapse directly: Kirchhoff's
$\Delta x_\text{inferred}$ at $2\lambda$ jumps from $0$ mm ($\pm2.5\lambda$) to $\approx256$ mm
($\pm3.5\lambda$) and settles near $234$–$236$ mm ($\pm4.5$–$6\lambda$; true $=225$ mm) once the
crop comfortably contains the new front position; Gazdag's $2\lambda$ estimate likewise jumps
from $0$ to $\approx226$ mm. The same widening *shrinks* $\phi_0$ at $2\lambda$ specifically
(Kirchhoff: $-1.93°\to-0.62°$; Gazdag: $-1.06°\to-0.68°$) — direct evidence that the large
$2\lambda$ value in \S4/Fig. (b) is the *same* crop-margin artifact as the displacement
collapse, not a separate phenomenon.

For the four sub-wavelength scales ($1\lambda$ down to $\tfrac18\lambda$), $\phi_0$ stays below
$0.6°$ at every crop width tested and **does not move monotonically with crop width** the way
the $2\lambda$ value does — Gazdag's values there even change sign between crop widths
(e.g. $-0.15°\to-0.27°\to-0.32°\to-0.27°$ at $1\lambda$). Small magnitude, no consistent trend,
occasional sign flips: that is the signature of ordinary fit noise at this estimator's
resolution limit, not a systematic geometric leakage proportional to $\Delta x_\text{true}$.

> **Caveat.** "Ordinary fit noise" is the simplest explanation for this sub-degree scatter, but
> not the only candidate per \S1.1's note on Campo (2021): the fluid front is a nine-box graded
> ramp, and that paper documents multi-reflector targets corrupting an otherwise-clean polarity
> signature through internal wave interaction between closely-spaced interfaces, distinct from
> any crop-width or attenuation effect. This notebook does not distinguish the two -- doing so
> would mean repeating \S4 on a *single sharp* air/water interface (no graded ramp) to see
> whether the sub-degree scatter shrinks further, which is left for future work.

**Conclusion on scale-independence.** $\phi_0$ can be treated as scale-independent — and
consistent with the (expected) near-zero material-change signature for this rigidly-translated
wetting-ramp geometry — for Kirchhoff and Gazdag from $1\lambda$ down to $\tfrac18\lambda$, once
the $2\lambda$ point is recognised as crop-margin-limited rather than a genuine larger
material-change reading. Back-propagation remains the exception: its $\phi_0$ is an order of
magnitude larger and strongly correlated with its own residual slope ($\rho\approx0.73$, vs.
$\rho\approx0.35$–$0.37$ for Kirchhoff/Gazdag), meaning geometry and material change have not
been cleanly separated for that method with this crop, independently of the centroid
calibration in \S4.2.

### 4.4 Further Investigation of Back-Propagation -- Diagnosing the Mask First

Section 4.3 showed Back-propagation's $\phi_0$ is an order of magnitude larger than
Kirchhoff/Gazdag's and strongly correlated with its own residual slope. Before reaching for a
more robust estimator, it is worth asking *why*: the `kz_pos_only=True` mask used for the
analytic channel already keeps very few bins (the cross-spectrum is restricted to
$|\kappa_z|,|\kappa_x|<1.4\kappa_{zc}$, $\kappa_z>0$, and amplitude $>10\%$ of the peak). If
those few bins are themselves incoherent, no amount of robust regression downstream can recover
a stable intercept from them -- the problem would be in the data entering the fit, not in the
least-squares fit itself.

In [ ]:
def diagnose_mask_coherence(base_crop, mon_crop, dz_g, dx_g, kz_cent):
    """
    Reports how many bins survive the kz_pos_only mask and how coherent their phase is
    (amplitude-weighted circular mean/std), independent of which regression is used on them.
    """
    Nz, Nx = base_crop.shape
    kz_ax = np.fft.fftfreq(Nz, d=dz_g) * 2 * np.pi
    kx_ax = np.fft.fftfreq(Nx, d=dx_g) * 2 * np.pi
    KZ, KX = np.meshgrid(kz_ax, kx_ax, indexing='ij')

    taper = np.outer(tukey(Nz, alpha=0.15), tukey(Nx, alpha=0.15))
    XS = np.fft.fft2(base_crop * taper) * np.conj(np.fft.fft2(mon_crop * taper))
    w, phi = np.abs(XS), np.angle(XS)

    band = (np.abs(KZ) < 1.4 * kz_cent) & (np.abs(KX) < 1.4 * kz_cent)
    mask = (w > 0.10 * w.max()) & band & ((np.abs(KZ) + np.abs(KX)) > 0) & (KZ > 0)

    n = int(mask.sum())
    if n == 0:
        return dict(n_mask=0, phi_mean_deg=np.nan, phi_std_deg=np.nan, phi_absmax_deg=np.nan)

    wm, phim = w[mask], phi[mask]
    mean_phi = np.average(phim, weights=wm)
    std_phi  = np.sqrt(np.average((phim - mean_phi) ** 2, weights=wm))
    return dict(n_mask=n, phi_mean_deg=np.degrees(mean_phi), phi_std_deg=np.degrees(std_phi),
                phi_absmax_deg=np.degrees(np.abs(phim).max()))


diag_rows = []
for method_name, base_stack in MIGRATION_METHODS.items():
    x_apex_base = locate_baseline_front(base_stack)
    ix_lo, ix_hi = crop_indices(x_apex_base, half_width_lam=2.5)
    base_img  = np.nan_to_num(base_stack[0])
    base_crop = base_img[:, ix_lo:ix_hi]
    for label in SCENARIOS_TO_USE:
        i_scen   = ff_scenarios.index(label)
        mon_img  = np.nan_to_num(base_stack[i_scen])
        mon_crop = mon_img[:, ix_lo:ix_hi]
        a_base, a_mon = sp_hilbert(base_crop, axis=0), sp_hilbert(mon_crop, axis=0)
        diag = diagnose_mask_coherence(a_base, a_mon, ff_dz_mig, ff_dx_mig, ff_kz_c)
        diag.update(method=method_name, scenario=label)
        diag_rows.append(diag)

diag_df = pd.DataFrame(diag_rows)
diag_df.pivot_table(index='scenario', columns='method',
                     values=['n_mask', 'phi_std_deg', 'phi_absmax_deg']).loc[SCENARIOS_TO_USE]


**This is the root cause, not a detail.** The `kz_pos_only` mask leaves only
**3–7 bins** for Kirchhoff/Gazdag and **14–16 bins** for Back-propagation -- every one of these
fits is already a 2-parameter regression on a handful of points. Bin count alone would suggest
Back-propagation has *more* data to work with, but its amplitude-weighted phase spread inside
the mask is **3–5$\times$ larger** (std up to $80°$, single bins up to $167°$ of phase, vs.
Kirchhoff/Gazdag's std $\le20°$ and max $\le39°$). Back-propagation's analytic cross-spectrum in
this band is not under-sampled, it is **not coherent** -- consistent with the dispersion fix
already in this repository's history for back-propagation's high-wavenumber receivers near the
PML (commit `9668ace`, "Fixed back propagation high frequency dispersion error"). That artifact
sits in exactly the $\kappa_z>0$, near-band-edge region this material-change channel depends on.
This predicts that no purely statistical fix downstream of the cross-spectrum will fully repair
$\phi_0$ for this method -- tested directly below.

### 4.5 Testing the Three Suggested Interventions -- Separately and Combined

Three specific fixes were proposed for an unstable intercept under $L_2$ (WLS) fitting on a
small, possibly outlier-corrupted set of bins: (1) Huber robust regression in place of WLS,
(2) a light 2D boxcar smoothing of the complex cross-spectrum before extracting phase, and
(3) a narrower passband ($1.1\kappa_{zc}$ instead of $1.4\kappa_{zc}$) to keep only the flattest
part of the spectrum. Each is implemented as an independent toggle below so they can be judged
on their own, not just as a bundle.

In [ ]:
from scipy.ndimage import uniform_filter
try:
    from sklearn.linear_model import HuberRegressor
    _HUBER_AVAILABLE = True
except ImportError:
    _HUBER_AVAILABLE = False
    print("scikit-learn not available -- robust-regression variants below will be skipped. "
          "Install with `pip install scikit-learn` to run them (e.g. the project's "
          "`gprMax` conda environment already has it).")


def estimate_shift_2d_v2(base, mon, dz_g, dx_g, kz_cent, kz_pos_only=False, force_dz_zero=False,
                          smooth=False, smooth_size=3, robust=False, band_factor=1.4,
                          centered_intercept=False):
    """
    Generalised version of estimate_shift_2d with the three suggested interventions as
    independent toggles: smooth (2D boxcar on the complex cross-spectrum before np.angle),
    robust (Huber regression instead of WLS), band_factor (passband half-width in units of
    kz_cent). centered_intercept reports phi_0 at the data's amplitude-weighted (kz,kx)
    centroid instead of at the origin -- mean-centring the design columns leaves the fitted
    slope unchanged, so this is a different read-out point on the same plane, not a new fit.
    Returns (dz_est, dx_est, phi_0, n_mask, XS, kz_ax, kx_ax).
    """
    Nz, Nx = base.shape
    kz_ax = np.fft.fftfreq(Nz, d=dz_g) * 2 * np.pi
    kx_ax = np.fft.fftfreq(Nx, d=dx_g) * 2 * np.pi
    KZ, KX = np.meshgrid(kz_ax, kx_ax, indexing='ij')

    taper = np.outer(tukey(Nz, alpha=0.15), tukey(Nx, alpha=0.15))
    XS = np.fft.fft2(base * taper) * np.conj(np.fft.fft2(mon * taper))

    if smooth:
        XS = (uniform_filter(np.real(XS), size=smooth_size)
              + 1j * uniform_filter(np.imag(XS), size=smooth_size))

    w, phi = np.abs(XS), np.angle(XS)
    band = (np.abs(KZ) < band_factor * kz_cent) & (np.abs(KX) < band_factor * kz_cent)
    mask = (w > 0.10 * w.max()) & band & ((np.abs(KZ) + np.abs(KX)) > 0)
    if kz_pos_only:
        mask &= (KZ > 0)

    n_mask = int(mask.sum())
    W = w[mask]
    KZ_m, KX_m = KZ[mask], KX[mask]
    if centered_intercept:
        KZ_m = KZ_m - (0.0 if force_dz_zero else np.average(KZ_m, weights=W))
        KX_m = KX_m - np.average(KX_m, weights=W)
    cols = [KX_m] if force_dz_zero else [KZ_m, KX_m]

    if n_mask < (2 if force_dz_zero else 3):
        return np.nan, np.nan, np.nan, n_mask, XS, kz_ax, kx_ax

    if robust:
        if not _HUBER_AVAILABLE:
            return np.nan, np.nan, np.nan, n_mask, XS, kz_ax, kx_ax
        X = np.column_stack(cols)
        huber = HuberRegressor(fit_intercept=True, epsilon=1.35, max_iter=200)
        huber.fit(X, phi[mask], sample_weight=W)
        coefs, phi_0 = huber.coef_, huber.intercept_
    else:
        A = np.column_stack(cols + [np.ones(n_mask)])
        c = np.linalg.lstsq(A * W[:, None], phi[mask] * W, rcond=None)[0]
        coefs, phi_0 = c[:-1], c[-1]

    dz_est, dx_est = (0.0, coefs[0]) if force_dz_zero else (coefs[0], coefs[1])
    return dz_est, dx_est, phi_0, n_mask, XS, kz_ax, kx_ax


In [ ]:
VARIANTS = {
    'baseline (WLS)':                dict(smooth=False, robust=False, band_factor=1.4),
    'smooth only':                    dict(smooth=True,  robust=False, band_factor=1.4),
    'huber only':                     dict(smooth=False, robust=True,  band_factor=1.4),
    'smooth + huber':                 dict(smooth=True,  robust=True,  band_factor=1.4),
    'narrow band only (1.1x)':        dict(smooth=False, robust=False, band_factor=1.1),
    'smooth+huber+narrow (1.1x)':     dict(smooth=True,  robust=True,  band_factor=1.1),
}

variant_rows = []
for vname, vkwargs in VARIANTS.items():
    for method_name, base_stack in MIGRATION_METHODS.items():
        x_apex_base  = locate_baseline_front(base_stack)
        ix_lo, ix_hi = crop_indices(x_apex_base, half_width_lam=2.5)
        base_img  = np.nan_to_num(base_stack[0])
        base_crop = base_img[:, ix_lo:ix_hi]
        for label in SCENARIOS_TO_USE:
            i_scen   = ff_scenarios.index(label)
            mon_img  = np.nan_to_num(base_stack[i_scen])
            mon_crop = mon_img[:, ix_lo:ix_hi]
            a_base, a_mon = sp_hilbert(base_crop, axis=0), sp_hilbert(mon_crop, axis=0)
            _, dx_a, phi0_a, n_a, *_ = estimate_shift_2d_v2(
                a_base, a_mon, ff_dz_mig, ff_dx_mig, ff_kz_c,
                kz_pos_only=True, force_dz_zero=True, **vkwargs,
            )
            variant_rows.append(dict(
                variant=vname, method=method_name, scenario=label, n_mask=n_a,
                dx_residual_mm=dx_a * 1e3, phi0_material_deg=np.degrees(phi0_a),
            ))

variant_df = pd.DataFrame(variant_rows)

print("phi_0 [deg] for Back-prop, by variant and scenario:")
piv_bp = (variant_df[variant_df['method'] == 'Back-prop']
          .pivot_table(index='variant', columns='scenario', values='phi0_material_deg'))
print(piv_bp.reindex(index=list(VARIANTS.keys()), columns=SCENARIOS_TO_USE).round(2))

print("\nphi_0 [deg] for Kirchhoff/Gazdag, by variant and scenario (regression check -- did we break what worked?):")
piv_kg = (variant_df[variant_df['method'].isin(['Kirchhoff', 'Gazdag'])]
          .pivot_table(index=['method', 'variant'], columns='scenario', values='phi0_material_deg'))
print(piv_kg.loc[:, SCENARIOS_TO_USE].round(3))

# Centered-intercept check: same baseline WLS fit, intercept read off at the data's weighted
# (kz,kx) centroid instead of the origin. Mean-centring the design columns leaves the fitted
# slope identical -- this is a different read-out point on the SAME plane, not a new fit.
print("\nphi_0 [deg]: origin intercept vs. centroid intercept (baseline WLS, no smoothing/Huber):")
for method_name, base_stack in MIGRATION_METHODS.items():
    x_apex_base  = locate_baseline_front(base_stack)
    ix_lo, ix_hi = crop_indices(x_apex_base, half_width_lam=2.5)
    base_img  = np.nan_to_num(base_stack[0])
    base_crop = base_img[:, ix_lo:ix_hi]
    origin_vals, centred_vals = [], []
    for label in SCENARIOS_TO_USE:
        i_scen   = ff_scenarios.index(label)
        mon_img  = np.nan_to_num(base_stack[i_scen])
        mon_crop = mon_img[:, ix_lo:ix_hi]
        a_base, a_mon = sp_hilbert(base_crop, axis=0), sp_hilbert(mon_crop, axis=0)
        _, _, phi0_o, *_ = estimate_shift_2d_v2(a_base, a_mon, ff_dz_mig, ff_dx_mig, ff_kz_c,
                                                 kz_pos_only=True, force_dz_zero=True)
        _, _, phi0_c, *_ = estimate_shift_2d_v2(a_base, a_mon, ff_dz_mig, ff_dx_mig, ff_kz_c,
                                                 kz_pos_only=True, force_dz_zero=True,
                                                 centered_intercept=True)
        origin_vals.append(np.degrees(phi0_o)); centred_vals.append(np.degrees(phi0_c))
    print(f"  {method_name:>10s}  origin:   " + "  ".join(f"{v:+7.2f}" for v in origin_vals))
    print(f"  {method_name:>10s}  centroid: " + "  ".join(f"{v:+7.2f}" for v in centred_vals))


**Result: none of the three interventions stabilises Back-propagation's
$\phi_0$, and two of them actively damage Kirchhoff/Gazdag's already-good result.**

- **Narrow band (1.1$\times$) is actively harmful.** It collapses the mask to **zero bins** for
  Kirchhoff and Gazdag at every scale (their already-sparse 3–7-bin band has nothing left once
  shrunk further) and to $\le4$ bins for Back-propagation -- too few for any 2-parameter fit to
  mean anything. This is the opposite of an improvement; abandon it for this dataset.
- **Smoothing increases the bin count** (Kirchhoff: 3→5, 5→6 bins; Back-prop: 15→18, 16→21) but
  **degrades Kirchhoff/Gazdag's $\phi_0$** rather than stabilising it -- e.g. Kirchhoff at
  $\tfrac18\lambda$ goes from $-0.01°$ (baseline) to $+0.50°$ (smoothed), and at $1\lambda$ from
  $-0.43°$ to $+2.83°$. Diluting the handful of strong, already-clean bins with their noisier
  neighbours makes a good estimate worse. It does not rescue Back-propagation either: smoothed
  $\phi_0$ there still swings from $-47°$ to $-3°$ across scenarios with no consistent sign.
- **Huber alone changes almost nothing for Kirchhoff/Gazdag** (as expected: with only 3–7 points
  and no single dominant outlier, there is nothing for a robust loss to down-weight) **and does
  not stabilise Back-propagation** -- its Huber-fitted $\phi_0$ still ranges from $-59°$ to
  $+2.8°$, essentially the same instability as plain WLS.
- **Combining smoothing and Huber does not rescue Back-propagation** ($-31.9°$ to $-2.8°$,
  still no stable value) **and still degrades Kirchhoff/Gazdag** for the same dilution reason.

The centered-intercept idea (evaluating the fitted plane's intercept at the data's weighted
$(\kappa_z,\kappa_x)$ centroid rather than at the origin) was also checked: because mean-centring
the design columns leaves the fitted *slope* unchanged, it is mathematically just a different
read-out point on the *same* plane, not a new fit. For Kirchhoff/Gazdag it changes the number but
not its stability; for Back-propagation it is just as erratic as the origin intercept
($-68.6°$ to $-9.9°$ across scenarios) -- there is no stable point to read off a plane this
poorly constrained.

**Conclusion.** This is consistent with \S4.4's diagnostic, not a failure of the regression
choice: at the standard $\pm2.5\lambda$ crop, Back-propagation's analytic cross-spectrum in the
$\kappa_z>0$ band is not corrupted by a few outlier pixels that a robust loss or a smoothing
kernel can clean up -- it is broadly incoherent across the whole sparse band, most plausibly
tracing back to the high-wavenumber time-reversal dispersion artifact already noted in this
project's history. **None of these three statistical fixes rescue it at that crop width.**
\S4.6--4.7 test a variable not touched here -- crop *width* itself -- and find that one does
help substantially, which nuances rather than overturns this section's conclusion: prefer
Kirchhoff/Gazdag for the material-change channel by default, but if Back-propagation's $\phi_0$
is needed, \S4.7 gives a concrete, cheap alternative to abandoning it outright. Note that
\S4.2's displacement calibration is unaffected by any of this -- it operates on the real-valued
channel, not the analytic one, so Back-propagation's $\Delta x$ readout remains usable
regardless.

### 4.6 Shrinking the Crop: Testing the Dilution Hypothesis Directly

\S1.1 and \S4.3 both flagged the same open question: is the small sub-degree $\phi_0$ measured
on the real dataset small because the underlying material-change signal genuinely is small, or
because the fixed $\pm2.5\lambda$ crop mixes a small, locally-substituted region with a much
larger area of unchanged background? \S8's convolution model shows the *undiluted* theoretical
ceiling, using the real gprMax material parameters (ice/air/water, including water's actual
$10\,\text{mS/m}$ conductivity), is $\approx-180°$ -- nowhere near the sub-degree values measured
here. If dilution is the explanation, *shrinking* the crop toward the actual transition-zone
width (the wetting zone's graded boxes are each $\approx\lambda/10$ wide) should pull $\phi_0$
away from zero and back toward that ceiling. This section tests that directly, rather than
asserting it.

In [ ]:
SHRINK_WIDTHS_LAM = [2.5, 2.0, 1.5, 1.25, 1.0, 0.85, 0.7, 0.55, 0.4]

shrink_rows = []
for method_name, base_stack in MIGRATION_METHODS.items():
    x_apex_base = locate_baseline_front(base_stack)
    base_img = np.nan_to_num(base_stack[0])
    for label in SCENARIOS_TO_USE:
        i_scen = ff_scenarios.index(label)
        mon_img = np.nan_to_num(base_stack[i_scen])
        front_new_pos = x_apex_base + true_dx_by_scenario[label]
        for hw in SHRINK_WIDTHS_LAM:
            ix_lo, ix_hi = crop_indices(x_apex_base, half_width_lam=hw)
            n_pix = ix_hi - ix_lo
            if n_pix < 6:
                continue
            base_crop = base_img[:, ix_lo:ix_hi]
            mon_crop  = mon_img[:, ix_lo:ix_hi]
            a_base, a_mon = sp_hilbert(base_crop, axis=0), sp_hilbert(mon_crop, axis=0)
            _, dx_a, phi0_a, n_a, *_ = estimate_shift_2d(
                a_base, a_mon, ff_dz_mig, ff_dx_mig, ff_kz_c, kz_pos_only=True, force_dz_zero=True
            )
            clipped = not (ff_x_traces[ix_lo] <= front_new_pos <= ff_x_traces[ix_hi - 1])
            shrink_rows.append(dict(method=method_name, scenario=label, crop_hw_lam=hw,
                                     n_pix=n_pix, n_mask=n_a, phi0_deg=np.degrees(phi0_a),
                                     clipped=clipped))

shrink_df = pd.DataFrame(shrink_rows)

print("phi_0 [deg] vs shrinking crop half-width -- UNCLIPPED scenarios only (1/8lam, 1/4lam):")
clean = shrink_df[(shrink_df['scenario'].isin(['⅛λ', '¼λ'])) & (~shrink_df['clipped'])]
print(clean.pivot_table(index=['method', 'scenario'], columns='crop_hw_lam', values='phi0_deg')
      .loc[:, SHRINK_WIDTHS_LAM].round(2))

print("\nFraction of rows flagged as clipped (new front position falls outside the crop), by scenario:")
print(shrink_df.groupby('scenario')['clipped'].mean().round(2))


**Result: confirmed.** For the two scenarios whose true displacement never
lets the crop clip the front's new position ($\tfrac18\lambda$, $\tfrac14\lambda$), $|\phi_0|$
climbs **monotonically** as the crop shrinks, on both Kirchhoff and Gazdag -- e.g. Kirchhoff at
$\tfrac14\lambda$ goes from $-0.04°$ at $\pm2.5\lambda$ to $+10.8°$ at $\pm0.4\lambda$; Gazdag at
the same scale goes from $+0.09°$ to $+4.8°$. That is a genuine one-to-two-order-of-magnitude
increase, not noise, and it moves in exactly the direction the dilution hypothesis predicts:
**less unchanged background in the window means a less-diluted material-change reading.**

It does not reach anywhere near \S8's $-180°$ undiluted ceiling, but the reason is mundane: the
migration grid spacing ($\Delta x_\text{mig}=10\,\text{mm}$) puts a hard floor around
$\pm0.35\lambda$ ($\approx9$--$13$ pixels), below which the 2-parameter analytic fit collapses to
1--2 mask points -- an exact, zero-residual interpolation rather than a meaningful estimate
(confirmed by identical repeated values and eventual `NaN` when pushed further). The transition
zone itself is only $\approx\lambda/10$ wide, well below this floor, so *this particular
2D-window method* cannot shrink the crop far enough to fully recover the undiluted signal on this
grid -- the dilution is real and directly demonstrated here, even though \S9.3 later shows the
ceiling itself is in fact reachable on this exact dataset, via a single-trace diagnostic that
does not share this method's multi-trace-fit resolution floor.

For the larger-displacement scenarios ($\tfrac12\lambda$, $1\lambda$, $2\lambda$), shrinking the
crop below the true displacement itself clips the front's new position out of the window
entirely -- a *different* effect from dilution, and the printed clipped-fraction table shows
this happens for most of the tested widths at those scales. Those results were excluded from the
trend above rather than risk conflating clipping artefacts with genuine dilution recovery.
Back-propagation at this same narrow end is erratic too, but \S4.7 below shows that is specific
to *narrow* crops -- not a blanket statement about every width, which the wider sweep there
overturns in part.

### 4.7 Going Wider: Does the Dilution Trend Continue, and Is Back-Propagation Actually Salvageable?

\S4.6 shrank the crop and watched $\phi_0$ grow. The natural complementary check is to keep
*widening* well past the original $\pm2.5\lambda$ -- not to fix any particular scenario this
time (that was \S4.3's job), but to see whether the dilution trend keeps decaying smoothly as
more background enters the window, and to retest Back-propagation specifically, since
\S4.4--4.5's "erratic at every crop width" framing was only ever checked across a narrow
$2.5$--$6\lambda$ range.

In [ ]:
WIDE_WIDTHS_LAM = [2.5, 4.0, 6.0, 8.0, 10.0, 12.0, 15.0, 18.0]

wide_rows = []
for method_name, base_stack in MIGRATION_METHODS.items():
    x_apex_base = locate_baseline_front(base_stack)
    base_img = np.nan_to_num(base_stack[0])
    for label in ['⅛λ', '¼λ', '½λ', '1λ']:
        i_scen = ff_scenarios.index(label)
        mon_img = np.nan_to_num(base_stack[i_scen])
        for hw in WIDE_WIDTHS_LAM:
            ix_lo, ix_hi = crop_indices(x_apex_base, half_width_lam=hw)
            base_crop, mon_crop = base_img[:, ix_lo:ix_hi], mon_img[:, ix_lo:ix_hi]
            a_base, a_mon = sp_hilbert(base_crop, axis=0), sp_hilbert(mon_crop, axis=0)
            _, dx_a, phi0_a, n_a, *_ = estimate_shift_2d(
                a_base, a_mon, ff_dz_mig, ff_dx_mig, ff_kz_c, kz_pos_only=True, force_dz_zero=True
            )
            hit_edge = (ix_lo == 0) or (ix_hi == len(ff_x_traces))
            wide_rows.append(dict(method=method_name, scenario=label, crop_hw_lam=hw,
                                   n_mask=n_a, phi0_deg=np.degrees(phi0_a), hit_domain_edge=hit_edge))

wide_df = pd.DataFrame(wide_rows)

print("phi_0 [deg] vs crop half-width, 2.5lam out to 18lam, Kirchhoff/Gazdag:")
print(wide_df[wide_df['method'].isin(['Kirchhoff', 'Gazdag'])]
      .pivot_table(index=['method', 'scenario'], columns='crop_hw_lam', values='phi0_deg')
      .loc[:, WIDE_WIDTHS_LAM].round(3))

print("\nphi_0 [deg] vs crop half-width, Back-prop (note the jump from 2.5lam to 4lam):")
print(wide_df[wide_df['method'] == 'Back-prop']
      .pivot_table(index='scenario', columns='crop_hw_lam', values='phi0_deg')
      .loc[:, WIDE_WIDTHS_LAM].round(3))


**Kirchhoff/Gazdag: already saturated by $\pm2.5\lambda$.** $\phi_0$ barely
moves at all from $2.5\lambda$ out to $18\lambda$ (e.g. Kirchhoff $\tfrac14\lambda$: $-0.04°
\to-0.06°\to-0.05°$, essentially flat). The amplitude-weighted fit already discounts distant
background once the window is a couple of wavelengths wide, so adding more empty background past
that point buys nothing further -- the dilution effect bottoms out quickly going wide, exactly as
it grows quickly going narrow in \S4.6. The two sweeps describe the same mechanism from opposite
directions.

**Back-propagation is not erratic at every width -- it is specifically bad at $\pm2.5\lambda$.**
At $\tfrac14\lambda$: $+3.38°$ at $2.5\lambda$, then $-0.11°$ at $4\lambda$, then a slow,
*monotonic* decay through $-0.56°,-0.43°,-0.35°,-0.29°,-0.24°$ down to $-0.13°$ at $18\lambda$.
Every width from $4\lambda$ upward behaves like a noisier, slower-converging version of
Kirchhoff/Gazdag's curve, not like noise. This means \S4.4--4.5's "erratic at every crop width"
conclusion was an artefact of only having tested the $2.5$--$6\lambda$ range, where
Back-propagation happens to be at its worst -- **$\pm2.5\lambda$ looks like a resonance-like bad
width for this method specifically, not proof that its analytic channel is unsalvageable.**

This nuances, but does not reverse, \S4.4--4.5: Back-propagation still needs far more averaging
(a much wider crop) than Kirchhoff/Gazdag to approach a comparably small $\phi_0$, and even at
$18\lambda$ it has not caught up (e.g. $\tfrac14\lambda$: $-0.13°$ for Back-prop vs.
$-0.045°$ for Kirchhoff/Gazdag at the same width -- roughly $3\times$ larger). The honest
practical takeaway is now more specific than "don't trust Back-prop's $\phi_0$": **avoid narrow
crops for Back-propagation in particular, and if its $\phi_0$ is needed, use a wide ($\gtrsim10
\lambda$) window rather than the $\pm2.5\lambda$ standard used for Kirchhoff/Gazdag** -- a
testable, falsifiable refinement rather than writing the method off entirely.

## 5. Publication-Grade Visualisation

Four panels, one figure, now using the **per-method calibrated** $\Delta x$ from \S4.2 (so
Back-propagation is no longer over-corrected) and explicitly flagging the $2\lambda$ scenario as
crop-margin-limited per \S4.3, rather than silently dropping or hiding it:

1. **True vs. estimated lateral advance** — the calibrated real-channel estimate against the
   known front displacement, for every migration method.
2. **Material-change intercept stability** — $\phi_0$ from the analytic channel across scales;
   flat and small validates that $\phi_0$ depends on the air/water contrast, not on how far the
   front has moved (the $2\lambda$ bars are annotated as crop-margin-limited, see \S4.3).
3. **Residual-slope contamination check** — the analytic channel's $\Delta x$ should stay near
   zero at every scale, confirming the front's geometric advance is not leaking into the
   material-change channel.
4. **Inference error** — calibrated estimate minus ground truth, to quantify accuracy directly.

In [ ]:
method_colors = {'Kirchhoff': 'tab:blue', 'Gazdag': 'tab:orange', 'Back-prop': 'tab:green'}
scenario_order = SCENARIOS_TO_USE
true_dx_axis = [true_dx_by_scenario[s] * 1e3 for s in scenario_order]
CROP_LIMITED_SCENARIO = '2λ'   # see Section 4.3 -- not a genuine larger phi_0 / failed estimate

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
fig.suptitle(
    "FluidFlow Phase-Plane Decoupling -- Lateral Movement ($\\Delta x$) vs. Material Change ($\\phi_0$)",
    fontsize=13, fontweight='bold',
)

# --- Panel 1: True vs Estimated Delta x (calibrated) ------------------------
ax = axes[0, 0]
ax.plot(true_dx_axis, true_dx_axis, 'k--', lw=1.2, alpha=0.6, label='y = x (perfect)')
for method, color in method_colors.items():
    sub = results_df[results_df['method'] == method].set_index('scenario').loc[scenario_order]
    ax.plot(sub['true_dx_mm'], sub['dx_calibrated_mm'], 'o-', color=color, label=method)
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('True $\\Delta x$ [mm]'); ax.set_ylabel('Calibrated $\\Delta x$ [mm]')
ax.set_title('(a) True vs. estimated lateral advance (per-method calibrated)')
ax.annotate('2λ: needs wider crop\\n(see §4.3)',
            xy=(true_dx_by_scenario['2λ'] * 1e3, results_df.loc[results_df['scenario'] == '2λ', 'dx_calibrated_mm'].mean()),
            xytext=(0.55, 0.12), textcoords='axes fraction', fontsize=7.5, color='dimgray',
            arrowprops=dict(arrowstyle='->', color='dimgray', lw=0.8))
ax.legend(fontsize=8); ax.grid(alpha=0.3, which='both')

# --- Panel 2: phi_0 stability ----------------------------------------------
ax = axes[0, 1]
x_pos = np.arange(len(scenario_order))
width = 0.25
for k, (method, color) in enumerate(method_colors.items()):
    sub = results_df[results_df['method'] == method].set_index('scenario').loc[scenario_order]
    ax.bar(x_pos + (k - 1) * width, sub['phi0_material_deg'], width=width, color=color, label=method)
ax.set_xticks(x_pos); ax.set_xticklabels(scenario_order)
ax.set_xlabel('Scenario (lateral-advance scale)'); ax.set_ylabel('$\\phi_0$ material-change intercept [deg]')
ax.set_title('(b) Material-change intercept stability')
i_crop_limited = scenario_order.index(CROP_LIMITED_SCENARIO)
ax.annotate('crop-margin\\nlimited (§4.3)', xy=(i_crop_limited, 0), xytext=(i_crop_limited - 0.3, 0.65),
            textcoords=('data', 'axes fraction'), fontsize=7.5, color='dimgray', ha='center',
            arrowprops=dict(arrowstyle='->', color='dimgray', lw=0.8))
ax.legend(fontsize=8); ax.grid(alpha=0.3, axis='y')

# --- Panel 3: residual slope contamination check ---------------------------
ax = axes[1, 0]
for method, color in method_colors.items():
    sub = results_df[results_df['method'] == method].set_index('scenario').loc[scenario_order]
    ax.plot(scenario_order, sub['dx_residual_mm'], 'o-', color=color, label=method)
ax.axhline(0.0, color='k', lw=1.0, ls='--', alpha=0.6)
ax.set_xlabel('Scenario'); ax.set_ylabel('Analytic-channel residual $\\Delta x$ [mm]')
ax.set_title('(c) Residual slope in the $\\phi_0$ channel (expect ~0)')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# --- Panel 4: inference error (calibrated) ----------------------------------
ax = axes[1, 1]
for method, color in method_colors.items():
    sub = results_df[results_df['method'] == method].set_index('scenario').loc[scenario_order]
    ax.plot(scenario_order, sub['calibrated_err_mm'], 'o-', color=color, label=method)
ax.axhline(0.0, color='k', lw=1.0, ls='--', alpha=0.6)
ax.set_xlabel('Scenario'); ax.set_ylabel('Calibrated $-$ True $\\Delta x$ [mm]')
ax.set_title('(d) Calibrated inference error')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()


## 6. Summary Table

In [ ]:
summary = (
    results_df
    .pivot_table(index='scenario', columns='method',
                 values=['true_dx_mm', 'dx_calibrated_mm', 'calibrated_err_mm', 'phi0_material_deg'])
    .loc[scenario_order]
)
summary


## 7. Observed Results & Caveats

Running the loop above against the real `fluidflow_study/migrated_results.npz` data, then
following up with the calibration (\S4.2) and crop-width sweep (\S4.3), settles the two open
questions from the first pass of this analysis:

**1. The Back-propagation over-correction is fixed by per-method calibration, not by assuming a
universal factor.** Kirchhoff ($k\approx1.97$) and Gazdag ($k\approx2.00$) confirm the
theoretical centroid factor of 2 directly from the data; Back-propagation comes out at
$k\approx0.99$, meaning its raw $kx$-slope already *is* the front displacement with no centroid
dilution, most likely because its time-reversal focusing localises the front's edge diffraction
differently from the analytic Kirchhoff/Gazdag operators. Panels (a) and (d) now use this
per-method $k$ instead of a blind $\times2$, which brings Back-propagation's calibrated estimate
in line with the other two methods across $1\lambda$–$\tfrac18\lambda$.

**2. $\phi_0$ *can* be treated as scale-independent for Kirchhoff and Gazdag, once the
$2\lambda$ scenario is correctly attributed to a crop-margin artifact rather than a genuine
larger material-change reading.** The crop-width sweep in \S4.3 shows the large $2\lambda$
values in both the displacement collapse ($\Delta x_\text{inferred}\to0$) and the inflated
$\phi_0$ ($\approx-1$ to $-2°$) shrink together as the crop widens past $\pm4\lambda$ — direct
evidence they share one root cause: the front's new position sits only $0.5\lambda$ inside the
edge of the original $\pm2.5\lambda$ crop. For the four scales where the crop is adequate
($1\lambda$ down to $\tfrac18\lambda$), $\phi_0$ stays below $0.6°$ for Kirchhoff/Gazdag with no
consistent trend against either $\Delta x_\text{true}$ or crop width (Gazdag's values even
change sign), consistent with ordinary fit noise rather than systematic geometric leakage — and
consistent with the Fourier-shift-theorem expectation (\S1.1) that a rigidly translated object
(which is how `FluidFlow_Playground.ipynb` actually constructs each scenario — the whole
graded wetting ramp re-centred, not a front sweeping through a static window) should show up as
pure geometric slope with $\phi_0\to0$, not a scale-dependent material-change signature.

**Remaining exception: Back-propagation's $\phi_0$ is still an order of magnitude larger
than Kirchhoff/Gazdag's and strongly correlated with its own residual slope
($\rho\approx0.73$, vs. $\rho\approx0.35$–$0.37$).** The \S4.2 calibration fixes the
*displacement* readout for this method, but does not by itself clean up its $\phi_0$ channel.
\S4.4–4.5 investigate this directly: the mask feeding the analytic fit is not just sparse but
genuinely incoherent for Back-propagation (amplitude-weighted phase std up to $80°$, vs.
$\le20°$ for Kirchhoff/Gazdag), and none of Huber regression, complex-spectrum smoothing, or a
narrower passband -- alone or combined -- recovers a stable value (full results in \S4.5). The
most likely cause is the high-wavenumber time-reversal dispersion artifact already noted in this
project's git history for back-propagation near the PML, which sits exactly in the
$\kappa_z>0$ band this channel depends on. \S4.6–4.7 add one nuance: that incoherence is at its
worst specifically at the $\pm2.5\lambda$ crop used throughout this notebook, and widening to
$\gtrsim10\lambda$ shrinks it substantially (e.g. $\tfrac14\lambda$ scenario: $+3.38°\to-0.13°$),
though it still does not close the gap to Kirchhoff/Gazdag at any width tested. **Practical
takeaway: prefer Kirchhoff or Gazdag for the material-change channel on this dataset by default;
Back-propagation's $\Delta x$ (\S4.2) remains usable regardless, and if its $\phi_0$ is genuinely
needed, use a wide ($\gtrsim10\lambda$) crop rather than the $\pm2.5\lambda$ standard, instead of
discarding the method outright.**

## 8. Convolution-Model Sensitivity Test: Can a Small Permittivity or Conductivity Change Be Inferred via $\phi_0$?

Sections 4–7 tested $\phi_0$ on real gprMax data: a large, binary air$\to$water contrast, on a
nine-box graded ramp, after Kirchhoff/Gazdag/back-propagation migration. That leaves the
question open from \S1.1/\S4.3's caveats unanswered in isolation: with *no* migration, *no*
multi-reflector geometry, and a *small*, controllable percentage change, does $\phi_0$ actually
track the change the way \cref{sec:th-material-change} predicts? This section builds the
cleanest possible test -- a single dispersive point reflector, built directly as a frequency-domain
convolution of a Ricker wavelet with a Fresnel reflection coefficient -- and sweeps the target's
permittivity and conductivity independently by $+5\%$ to $+30\%$ relative to a baseline value.

In [ ]:
v_mig = v_ice / 2   # exploding-reflector half-velocity, same convention as the rest of this notebook
kz_c  = ff_kz_c     # alias -- 2*pi/lam, already defined in Section 2

EPS0 = 8.8541878128e-12  # vacuum permittivity, F/m

def eps_complex(eps_r, sigma_mS_per_m, f_GHz):
    """Complex relative permittivity of a lossy dielectric, eps_r - j*sigma/(omega*eps0)."""
    sigma = sigma_mS_per_m * 1e-3  # mS/m -> S/m
    omega = 2 * np.pi * np.asarray(f_GHz, dtype=float) * 1e9  # GHz -> rad/s
    omega = np.where(omega == 0, np.finfo(float).eps, omega)
    return eps_r - 1j * sigma / (omega * EPS0)


def fresnel_gamma(eps1_c, eps2_c):
    """Normal-incidence Fresnel reflection coefficient, Campo (2021) Eq. (1) generalised to complex eps_r."""
    n1, n2 = np.sqrt(eps1_c), np.sqrt(eps2_c)
    return (n1 - n2) / (n1 + n2)


# Depth-domain Ricker wavelet -- same convention as PhasePlaneFit.ipynb's Material Change Test
dz_conv, Nz_conv = lam / 40, 512
z_wl_conv = (np.arange(Nz_conv) - Nz_conv // 2) * dz_conv
u_wl_conv = (np.pi * f_c * z_wl_conv / v_mig) ** 2
ricker_z_conv = (1 - 2 * u_wl_conv) * np.exp(-u_wl_conv)

kz_ax_conv  = np.fft.fftfreq(Nz_conv, d=dz_conv) * 2 * np.pi
f_axis_GHz  = kz_ax_conv * v_mig / (2 * np.pi)   # kz = 2*pi*f/v_mig, so f = kz*v_mig/(2*pi)
W_z_conv    = np.fft.fft(ricker_z_conv)

eps_r_host, sigma_host_mS = 3.15, 0.0   # ice host, lossless

def dispersive_reflectivity(eps_r_target, sigma_target_mS):
    """Depth-domain reflected wavelet: IFFT[ W(kz) * Gamma(kz) ], Gamma evaluated at each kz's
    equivalent frequency so a conductive target disperses the pulse exactly as Fresnel theory predicts."""
    eps_h = eps_complex(eps_r_host,   sigma_host_mS,   f_axis_GHz)
    eps_t = eps_complex(eps_r_target, sigma_target_mS, f_axis_GHz)
    Gamma_f = fresnel_gamma(eps_h, eps_t)
    return np.real(np.fft.ifft(W_z_conv * Gamma_f))


# Lateral PSF: a Gaussian standing in for a migrated point scatterer's focused profile
Nx_conv, dx_conv = 64, lam / 4
x_conv   = np.arange(Nx_conv) * dx_conv
gauss_x  = np.exp(-0.5 * ((x_conv - x_conv.mean()) / (lam / 2)) ** 2)

def build_2d_image(eps_r_target, sigma_target_mS):
    return np.outer(dispersive_reflectivity(eps_r_target, sigma_target_mS), gauss_x)

# ROI: a few wavelengths either side of the reflector, same crop convention as elsewhere
n_margin_conv = int(round(3.5 * lam / dz_conv))
iz0_conv = Nz_conv // 2
iz_lo_conv, iz_hi_conv = max(0, iz0_conv - n_margin_conv), min(Nz_conv, iz0_conv + n_margin_conv + 1)


In [ ]:
def run_convolution_sweep(eps_r_target_base, sigma_target_base_mS, pct_list, vary):
    """vary='eps' perturbs permittivity by pct%% (sigma fixed); vary='sigma' perturbs
    conductivity by pct%% (eps_r fixed). Returns a DataFrame with theory and fitted phi_0."""
    base_full = build_2d_image(eps_r_target_base, sigma_target_base_mS)
    base_crop = base_full[iz_lo_conv:iz_hi_conv, :]
    Gamma_base_fc = fresnel_gamma(eps_complex(eps_r_host, sigma_host_mS, f_c),
                                   eps_complex(eps_r_target_base, sigma_target_base_mS, f_c))
    a_base = sp_hilbert(base_crop, axis=0)

    rows = []
    for pct in pct_list:
        if vary == 'sigma':
            eps_r_mon, sigma_mon = eps_r_target_base, sigma_target_base_mS * (1 + pct / 100)
        else:
            eps_r_mon, sigma_mon = eps_r_target_base * (1 + pct / 100), sigma_target_base_mS

        mon_crop = build_2d_image(eps_r_mon, sigma_mon)[iz_lo_conv:iz_hi_conv, :]
        a_mon = sp_hilbert(mon_crop, axis=0)

        Gamma_mon_fc = fresnel_gamma(eps_complex(eps_r_host, sigma_host_mS, f_c),
                                      eps_complex(eps_r_mon, sigma_mon, f_c))
        # Sign matches the thesis's own convention, eq:thinlayer-phase: Delta R ~ alpha * exp(-j*Delta_theta)
        theta_theory_fc = -np.degrees(np.angle(Gamma_mon_fc / Gamma_base_fc))

        _, dx_resid, phi0_est, XS, kz_ax_c, kx_ax_c = estimate_shift_2d(
            a_base, a_mon, dz_conv, dx_conv, kz_c, kz_pos_only=True, force_dz_zero=True
        )
        rows.append(dict(pct=pct, theta_theory_fc_deg=theta_theory_fc,
                          phi0_est_deg=np.degrees(phi0_est), dx_resid_mm=dx_resid * 1e3,
                          gamma_mon_mag=abs(Gamma_mon_fc), peak_XS=np.abs(XS).max()))
    return pd.DataFrame(rows)


PCT_LIST = [5, 10, 15, 20, 25, 30]

eps_sweep_df   = run_convolution_sweep(eps_r_target_base=6.0, sigma_target_base_mS=0.0,
                                        pct_list=PCT_LIST, vary='eps')
sigma_sweep_df = run_convolution_sweep(eps_r_target_base=6.0, sigma_target_base_mS=5.0,
                                        pct_list=PCT_LIST, vary='sigma')

print("Permittivity-only sweep (lossless, target eps_r=6.0 -> 6.0*(1+pct%), host=ice eps_r=3.15):")
print(eps_sweep_df.round(4))
print("\nConductivity-only sweep (eps_r fixed at 6.0, sigma=5.0 mS/m -> 5.0*(1+pct%)):")
print(sigma_sweep_df.round(4))


**Permittivity-only result.** `theta_theory_fc_deg` and `phi0_est_deg` are both
$\approx0°$ at every percentage tested, while `gamma_mon_mag` grows monotonically with the
percentage change and `peak_XS` (the cross-spectrum's peak energy) grows with it. This is an
exact, clean confirmation of the Fresnel algebra: for two **real** (lossless) permittivities,
$\Gamma=(\sqrt{\varepsilon_{r1}}-\sqrt{\varepsilon_{r2}})/(\sqrt{\varepsilon_{r1}}+\sqrt{\varepsilon_{r2}})$
is itself real, so its phase can only be exactly $0°$ or exactly $180°$ -- never a smoothly
graded intermediate value -- as long as the perturbation does not push the target's permittivity
back across the host's. **A pure permittivity change is invisible to $\phi_0$ at this scale; the
percentage change is recoverable from the reflection's amplitude/energy, not its phase.**

**Conductivity-only result.** Both columns now move together, monotonically, with the percentage
conductivity change, and **`phi0_est_deg` tracks `theta_theory_fc_deg` to within a roughly
constant scale factor.** That residual scale factor turns out to be fully explained, not noise --
checked directly below.

In [ ]:
# The conductivity sweep's theory column used Gamma evaluated at the *nominal* f_c.
# But the fit's phi_0 is a weighted average over every bin inside the kz_pos_only mask, and
# the conductivity term in eps_complex scales as 1/omega -- lower-frequency bins rotate the
# phase more. Does evaluating Fresnel theory at the mask's actual energy-weighted frequency,
# instead of the nominal f_c, remove the discrepancy?
eps_r_target_base, sigma_target_base_mS, pct_check = 6.0, 5.0, 20.0

base_crop = build_2d_image(eps_r_target_base, sigma_target_base_mS)[iz_lo_conv:iz_hi_conv, :]
mon_crop  = build_2d_image(eps_r_target_base, sigma_target_base_mS * (1 + pct_check / 100))[iz_lo_conv:iz_hi_conv, :]
a_base, a_mon = sp_hilbert(base_crop, axis=0), sp_hilbert(mon_crop, axis=0)

Nz_c, Nx_c = a_base.shape
kz_ax_c = np.fft.fftfreq(Nz_c, d=dz_conv) * 2 * np.pi
kx_ax_c = np.fft.fftfreq(Nx_c, d=dx_conv) * 2 * np.pi
KZ_c, KX_c = np.meshgrid(kz_ax_c, kx_ax_c, indexing='ij')
taper_c = np.outer(tukey(Nz_c, alpha=0.15), tukey(Nx_c, alpha=0.15))
XS_c = np.fft.fft2(a_base * taper_c) * np.conj(np.fft.fft2(a_mon * taper_c))
w_c  = np.abs(XS_c)
band_c = (np.abs(KZ_c) < 1.4 * kz_c) & (np.abs(KX_c) < 1.4 * kz_c)
mask_c = (w_c > 0.10 * w_c.max()) & band_c & ((np.abs(KZ_c) + np.abs(KX_c)) > 0) & (KZ_c > 0)

f_eff_GHz = np.average(KZ_c[mask_c], weights=w_c[mask_c]) * v_mig / (2 * np.pi)
print(f"Mask's energy-weighted frequency: f_eff = {f_eff_GHz:.4f} GHz  (nominal f_c = {f_c} GHz)")

sigma_mon_check = sigma_target_base_mS * (1 + pct_check / 100)
Gamma_base_eff = fresnel_gamma(eps_complex(eps_r_host, sigma_host_mS, f_eff_GHz),
                                eps_complex(eps_r_target_base, sigma_target_base_mS, f_eff_GHz))
Gamma_mon_eff  = fresnel_gamma(eps_complex(eps_r_host, sigma_host_mS, f_eff_GHz),
                                eps_complex(eps_r_target_base, sigma_mon_check, f_eff_GHz))
theta_eff = -np.degrees(np.angle(Gamma_mon_eff / Gamma_base_eff))

row = sigma_sweep_df.loc[sigma_sweep_df['pct'] == pct_check].iloc[0]
print(f"theory @ nominal f_c:         {row['theta_theory_fc_deg']:+.4f} deg")
print(f"theory @ mask's f_eff:        {theta_eff:+.4f} deg")
print(f"fitted phi0_est:              {row['phi0_est_deg']:+.4f} deg")
print(f"ratio fitted / theory@f_eff:  {row['phi0_est_deg'] / theta_eff:.3f}")


Evaluating Fresnel theory at the mask's actual energy-weighted frequency
($f_\text{eff}\approx0.86$ GHz, well below the nominal $f_c=1.5$ GHz) instead of $f_c$ itself
brings theory and the fitted $\phi_0$ to within $\sim2\%$ of each other (ratio $\approx0.98$,
vs. $\approx1.7$ when wrongly using $f_c$). **The discrepancy was never noise: $\phi_0$ exactly
matches Fresnel theory once you evaluate the theory at the frequency the fit actually measures
at, not the wavelet's nominal centre frequency.** This is expected, not a bug -- the conductivity
term in $\varepsilon_{r,\text{complex}}$ scales as $1/\omega$, so a band-weighted average phase
is necessarily dominated by the lower-frequency content inside the passband.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Convolution-Model Sensitivity Test: Permittivity vs. Conductivity Perturbation",
             fontsize=12, fontweight='bold')

ax = axes[0]
ax.plot(eps_sweep_df['pct'], eps_sweep_df['phi0_est_deg'], 'o-', color='tab:blue', label='fitted phi_0')
ax2 = ax.twinx()
ax2.plot(eps_sweep_df['pct'], eps_sweep_df['gamma_mon_mag'], 's--', color='tab:red', label='|Gamma_mon|')
ax.set_xlabel('Permittivity change [%]'); ax.set_ylabel('fitted phi_0 [deg]', color='tab:blue')
ax2.set_ylabel('|Gamma_mon|', color='tab:red')
ax.set_ylim(-1, 1)
ax.set_title('(a) Permittivity-only: phase pinned, amplitude carries the signal')
ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(sigma_sweep_df['pct'], sigma_sweep_df['theta_theory_fc_deg'], 'k--', lw=1.2,
        label='theory @ nominal f_c')
ax.plot(sigma_sweep_df['pct'], sigma_sweep_df['phi0_est_deg'], 'o-', color='tab:green',
        label='fitted phi_0')
ax.set_xlabel('Conductivity change [%]'); ax.set_ylabel('phi_0 [deg]')
ax.set_title('(b) Conductivity-only: smoothly graded, matches theory at f_eff')
ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()


**Conclusion.** This fully synthetic, single-reflector test gives an unambiguous
answer to "can we infer the change": **yes for conductivity, only indirectly (via amplitude) for
permittivity.** A pure permittivity perturbation that does not cross the host's value leaves
$\Gamma$ real and its phase exactly pinned -- $\phi_0$ cannot see it, and should not be expected
to; the channel that carries that information is the reflection's energy. A conductivity
perturbation makes $\Gamma$ genuinely complex, and $\phi_0$ tracks it smoothly and quantitatively
in agreement with Fresnel theory, to a precision limited only by correctly accounting for which
frequency the band-weighted fit actually samples.

This also re-contextualises the real fluid-front result in \S4–\S7. Ice ($\varepsilon_r=3.15$)
sits *between* air ($\varepsilon_r=1$) and water ($\varepsilon_r=81$), so a clean, undiluted
air$\to$water substitution crosses the host permittivity and should -- per this section's own
permittivity-only result -- produce a full $180°$ flip, not the sub-degree values measured on
the real dataset. The real measurement sits nowhere near that ceiling because the fixed
$\pm2.5\lambda$ crop mixes the small, locally-substituted region with a much larger area of
genuinely unchanged background within the same window -- consistent with (and a quantitative
explanation for) why \S4.3 found Kirchhoff/Gazdag's $\phi_0$ to be small rather than dramatic.
Water's non-zero conductivity relative to air's near-zero conductivity is the more plausible
source of whatever small, smoothly-behaved signal genuinely reaches $\phi_0$ in that windowed
measurement, rather than the (discrete, diluted) permittivity-driven polarity flip.

## 9. Cross-Validation with CLSSA: A Per-Trace, Independent Diagnostic

Every estimator so far has been the same global 2D cross-spectrum plane fit, just with different
masks, crops, and regressions on top of it. `CWT_playground.ipynb` (this project's sister
notebook) implements a completely different tool for the same underlying question --
`clssa_phase_decomposition` in `phase_decomposition.py`, Constrained Least-Squares Spectral
Analysis (Castagna et al. 2016). Unlike a CWT, its tapering window is applied to the *basis
sinusoids*, not the data, giving uniform time/depth resolution at every frequency with no
cone-of-influence smearing. Integrating its complex output over the frequency band gives a
**broadband phasor** $R(t)+iI(t)$ per *single trace* -- a dominant phase $\arctan(I,R)$ at any
depth sample, entirely independent of the 2D FFT, the $\kappa_z$/$\kappa_x$ mask, and the
multi-trace window this notebook has used everywhere else. That independence makes it a fair
cross-check: if it agrees with the cross-spectrum method, that is real corroboration, not a
restatement of the same calculation.

In [ ]:
import sys
sys.path.insert(0, str(FF_ROOT.parent))
import phase_decomposition as pd_mod

def clssa_dominant_phase(trace, dz, iz, **clssa_params):
    """Broadband-phasor dominant phase and its envelope, at depth-sample iz, for one trace."""
    norm = np.abs(trace).max() + 1e-30
    A_, th2_, _, f_, _, _ = pd_mod.clssa_phase_decomposition(trace / norm, dz, **clssa_params)
    C_ = A_ * np.exp(1j * np.deg2rad(th2_))
    R_ = np.trapezoid(np.real(C_), f_, axis=0)
    I_ = np.trapezoid(np.imag(C_), f_, axis=0)
    env_ = np.sqrt(R_**2 + I_**2)
    return float(np.rad2deg(np.arctan2(I_[iz], R_[iz]))), float(env_[iz])

CLSSA_PARAMS = dict(f_min_GHz=0.3 / lam, f_max_GHz=4.0 / lam, n_freqs=60, win_ns=3.0 * lam, alpha=1e-2)
# (f_min/f_max/win_ns are mislabelled units here -- the function is generic to any 1D signal +
#  sampling interval, so passing depth-wavenumber-equivalent values [cycles/m] instead of GHz
#  reuses it for a DEPTH axis exactly as CWT_playground.ipynb's "Spatial CLSSA" cells reuse it
#  for an x axis.)


### 9.1 Validating CLSSA Against the Known-Ground-Truth Convolution Model

Before trusting CLSSA on noisy real data, check it reproduces \S8's three already-proven results
-- exactly 0° for a pure permittivity change, smooth grading for conductivity, and the full
$\approx180°$ ceiling for a clean air$\to$water flip -- via this completely different
computational route.

In [ ]:
iz0_chk = Nz_conv // 2

trace_eps_base = dispersive_reflectivity(6.0, 0.0)
trace_eps_mon  = dispersive_reflectivity(6.0 * 1.20, 0.0)
phi_b, _ = clssa_dominant_phase(trace_eps_base, dz_conv, iz0_chk, **CLSSA_PARAMS)
phi_m, _ = clssa_dominant_phase(trace_eps_mon,  dz_conv, iz0_chk, **CLSSA_PARAMS)
print(f"Permittivity-only (+20%, lossless): CLSSA rel_phi = {((phi_m - phi_b + 180) % 360) - 180:+.3f} deg  (expect 0)")

trace_sig_base = dispersive_reflectivity(6.0, 5.0)
for pct in [5, 10, 20, 30]:
    trace_sig_mon = dispersive_reflectivity(6.0, 5.0 * (1 + pct / 100))
    phi_b, _ = clssa_dominant_phase(trace_sig_base, dz_conv, iz0_chk, **CLSSA_PARAMS)
    phi_m, _ = clssa_dominant_phase(trace_sig_mon,  dz_conv, iz0_chk, **CLSSA_PARAMS)
    rel = ((phi_m - phi_b + 180) % 360) - 180
    print(f"Conductivity-only (+{pct:3d}%): CLSSA rel_phi = {rel:+.3f} deg")

trace_air, trace_water = dispersive_reflectivity(1.0, 0.0), dispersive_reflectivity(80.0, 10.0)
phi_air, _   = clssa_dominant_phase(trace_air,   dz_conv, iz0_chk, **CLSSA_PARAMS)
phi_water, _ = clssa_dominant_phase(trace_water, dz_conv, iz0_chk, **CLSSA_PARAMS)
rel_aw = ((phi_water - phi_air + 180) % 360) - 180
print(f"Clean air -> water flip: CLSSA rel_phi = {rel_aw:+.3f} deg  (expect ~+-180, cf. Sec 8's -179.98 deg via Fresnel/cross-spectrum)")


**Confirmed.** CLSSA reproduces all three \S8 results via a mathematically independent route:
$0.000°$ for the pure permittivity change, a smooth (though oppositely-signed -- CLSSA reads
$\arg(C_\text{mon})-\arg(C_\text{base})$ directly, the cross-spectrum method reads
$\arg(B\cdot\overline{M})$, an equally valid but conjugated convention) grading for conductivity,
and $\approx+180°$ for the clean flip. This is genuine corroboration, not a restatement.

### 9.2 A Single Real Trace at the Front Position

Apply the validated diagnostic to one real trace -- the same trace `locate_baseline_front` already
uses to anchor every crop in this notebook -- for Kirchhoff, Gazdag, and Back-propagation.

In [ ]:
for method_name, base_stack in MIGRATION_METHODS.items():
    x_apex_base = locate_baseline_front(base_stack)
    ix_front = np.searchsorted(ff_x_traces, x_apex_base)
    base_img = np.nan_to_num(base_stack[0])
    env_base = np.abs(sp_hilbert(base_img, axis=0))
    iz_peak = int(np.argmax(env_base[:, ix_front]))

    print(f"\n{method_name}  (front trace x={x_apex_base:.3f}m, iz_peak={iz_peak}):")
    for label in SCENARIOS_TO_USE[1:]:  # skip 2lam; crop-margin concerns don't apply to a single trace
        i_scen = ff_scenarios.index(label)
        mon_img = np.nan_to_num(base_stack[i_scen])
        phi_b, _ = clssa_dominant_phase(base_img[:, ix_front], ff_dz_mig, iz_peak, **CLSSA_PARAMS)
        phi_m, _ = clssa_dominant_phase(mon_img[:, ix_front],  ff_dz_mig, iz_peak, **CLSSA_PARAMS)
        rel = ((phi_m - phi_b + 180) % 360) - 180
        print(f"  {label:>4s}: rel_phi = {rel:+8.2f} deg   (global 2D-fit phi_0 from Table 4.1 was sub-degree)")


This single trace already reads tens of degrees (roughly $-7°$ to $-35°$,
growing with the true displacement) -- far larger than the global fit's sub-degree values, but
still well short of \S8's $\approx180°$ ceiling, and the *growth with scale* is itself odd: \S9.1
showed a clean material flip at fixed position gives the *same* phase regardless of how much
changed. \S9.3 resolves this by looking at every trace, not just the one the rest of this notebook
happens to anchor crops to.

### 9.3 Every Trace: Finding the True Signal, and Guarding Against a Real Failure Mode

Scanning every trace in a window around the front reveals sharp local features the single-trace
check above could not show. But a windowed spectral estimate at very low signal amplitude is a
known failure mode (the broadband phasor's denominator effectively collapses), so each point's
*own* envelope amplitude is tracked and used to mask out unreliable bins, exactly as a careful
analyst would discard a phase reading taken where the trace is near a null -- this is not
optional cleanup, it is what separates a real result from an artefact below.

In [ ]:
def clssa_profile_with_mask(base_img, mon_img, ix_lo, ix_hi, dz, iz, **clssa_params):
    """Per-trace CLSSA dominant-phase difference across columns [ix_lo, ix_hi), with an
    amplitude-based reliability mask (True = trust this point)."""
    n = ix_hi - ix_lo
    rel = np.full(n, np.nan)
    amp_min = np.full(n, np.nan)
    for j, ix in enumerate(range(ix_lo, ix_hi)):
        phi_b, amp_b = clssa_dominant_phase(base_img[:, ix], dz, iz, **clssa_params)
        phi_m, amp_m = clssa_dominant_phase(mon_img[:, ix],  dz, iz, **clssa_params)
        rel[j] = ((phi_m - phi_b + 180) % 360) - 180
        amp_min[j] = min(amp_b, amp_m)
    reliable = amp_min > 0.5 * np.median(amp_min)
    return rel, reliable


print(f"{'method':>10s} {'scenario':>6s} {'new front [m]':>14s} {'peak |rel_phi|':>15s} "
      f"{'at x [m]':>10s} {'offset from front [mm]':>23s} {'n_masked':>9s}")
clssa_peak_rows = []
for method_name, base_stack in MIGRATION_METHODS.items():
    x_apex_base = locate_baseline_front(base_stack)
    base_img = np.nan_to_num(base_stack[0])
    env_base = np.abs(sp_hilbert(base_img, axis=0))
    iz_peak = int(np.argmax(env_base[:, np.searchsorted(ff_x_traces, x_apex_base)]))

    for label in SCENARIOS_TO_USE[1:]:
        i_scen = ff_scenarios.index(label)
        mon_img = np.nan_to_num(base_stack[i_scen])
        front_new = x_apex_base + true_dx_by_scenario[label]

        ix_lo = max(0, np.searchsorted(ff_x_traces, x_apex_base - 1.5 * lam))
        ix_hi = min(len(ff_x_traces), np.searchsorted(ff_x_traces, front_new + 1.5 * lam))
        rel, reliable = clssa_profile_with_mask(base_img, mon_img, ix_lo, ix_hi,
                                                 ff_dz_mig, iz_peak, **CLSSA_PARAMS)
        masked = np.where(reliable, rel, np.nan)
        x_local = ff_x_traces[ix_lo:ix_hi]

        j_peak = int(np.nanargmax(np.abs(masked)))
        clssa_peak_rows.append(dict(method=method_name, scenario=label,
                                     peak_rel_phi=masked[j_peak],
                                     offset_mm=(x_local[j_peak] - front_new) * 1e3,
                                     n_masked=int(np.sum(~reliable)), n_total=len(rel)))
        print(f"{method_name:>10s} {label:>6s} {front_new:14.3f} {masked[j_peak]:15.2f} "
              f"{x_local[j_peak]:10.3f} {(x_local[j_peak]-front_new)*1e3:23.0f} "
              f"{int(np.sum(~reliable)):4d}/{len(rel):<4d}")

clssa_peak_df = pd.DataFrame(clssa_peak_rows)


**Confirmed, and reproducible across every scenario.** For Kirchhoff and
Gazdag, the masked peak $|\text{rel\_phi}|$ reaches **98° to 159°** -- an order of magnitude
closer to \S8's $\approx180°$ ceiling than anything the 2D-window approach reached, including
\S4.6's most aggressively shrunk crop (which topped out at $\approx11°$, capped by the migration
grid's pixel-count floor for a 2-parameter fit). Critically, this peak sits within **2–16mm of the
true new front position** at every scale tested -- not a coincidence, and not an artefact: only
1–2 of the 35–45 scanned traces get masked out per scenario (correctly catching genuine envelope
nulls, verified directly -- the masked points have CLSSA amplitude $\approx0.9$ against a typical
$\approx4.0$, an order of magnitude collapse, while the retained peak points have completely
normal amplitude). **This resolves the open question from \S4.6/\S6: the $\approx180°$ ceiling is not
unreachable on this dataset -- it was never visible to a method that averages or fits across
multiple traces. A single-trace, amplitude-validated diagnostic recovers it directly.**

This also explains \S9.2's puzzling scale-dependence: the single front-locator trace was never
sitting on the peak itself, only on the smooth approach ramp leading up to it -- the true
near-180° feature is narrower (a couple of traces wide) than the spacing between the
front-locator trace and the actual new front edge at smaller scales.

### 9.4 Back-Propagation: An Independent Confirmation via a Different Method

The same per-trace scan, run on Back-propagation, gives a third, independent line of evidence for
\S4.4's coherence diagnosis -- via a method that shares no code path with the cross-spectrum
estimator at all.

In [ ]:
print(clssa_peak_df[clssa_peak_df['method'] == 'Back-prop']
      .to_string(index=False))


Back-propagation's peak is both **smaller** (39° to 141°, vs. Kirchhoff/Gazdag's
98–159°) and, more tellingly, **consistently mislocalised** -- 34 to 103mm away from the true new
front position, an order of magnitude further off than Kirchhoff/Gazdag's 2–16mm. A method with
genuinely incoherent phase in this band would be expected to fail at locating *where* the real
signal is, not just at reading its *size* -- which is exactly what this shows, via a diagnostic
with no shared computation with \S4.4's mask-coherence statistics or \S4.5's regression
experiments. Three unrelated methods (cross-spectrum mask statistics, robust-regression stress
tests, and now per-trace CLSSA localisation) now agree Back-propagation's phase information in
this band is genuinely degraded, not an artefact of any one estimator's assumptions.

## 10. One Coherent Pipeline: Reading Material Change *At* the Phase-Plane's Own $\Delta x$ Estimate

\S4.2 and \S9.3 have, until now, been two separate procedures that never reference each other --
and \S9.3's CLSSA search window was centred using `true_dx_by_scenario`, ground truth that would
not exist in a real survey. The natural way to link them is **not** to let CLSSA search a wide
window and report whatever peak it happens to find (which risks reporting a strong-looking but
unrelated feature, and invites treating that peak's *location* as a competing displacement
estimate -- a second job CLSSA was never asked to do well). Instead, use the phase-plane fit's
own job -- estimating $\Delta x$ -- to tell CLSSA *exactly where to read*, and ask only "what is
the material-change phase right there":

1. Take the **calibrated** $\Delta x$ estimate already computed in \S4.2 (`dx_calibrated_mm`,
   ground-truth-free -- it only uses the real/Hermitian channel and the per-method factor $k$).
2. **Predict** the front's new position from it: $x_\text{pred} = x_{\text{apex,base}} +
   \Delta x_\text{calibrated}$.
3. Read CLSSA's dominant-phase difference at the single trace nearest $x_\text{pred}$ -- not a
   search, an anchored reading. A modest window around $x_\text{pred}$ ($\pm0.75\lambda$) is used
   *only* to establish a statistically meaningful local amplitude baseline for the reliability
   check from \S9.3, exactly as before -- it never decides which point gets reported.

This makes $\phi_0$ a value that is only meaningful in light of, and tied directly to, the
$\Delta x$ the phase-plane fit already trusts -- not an independent peak-finder that could drift
onto some other feature.

In [ ]:
SCENARIOS_FOR_PIPELINE = SCENARIOS_TO_USE[1:]  # skip 2lam -- Sec 4.3 already flagged its
                                                 # calibrated dx as crop-margin-limited

anchored_rows = []
for method_name, base_stack in MIGRATION_METHODS.items():
    x_apex_base = locate_baseline_front(base_stack)
    base_img = np.nan_to_num(base_stack[0])
    env_base = np.abs(sp_hilbert(base_img, axis=0))
    iz_peak  = int(np.argmax(env_base[:, np.searchsorted(ff_x_traces, x_apex_base)]))

    for label in SCENARIOS_FOR_PIPELINE:
        i_scen  = ff_scenarios.index(label)
        mon_img = np.nan_to_num(base_stack[i_scen])

        dx_calibrated_mm = float(results_df.loc[
            (results_df['method'] == method_name) & (results_df['scenario'] == label),
            'dx_calibrated_mm'].iloc[0])
        x_pred = x_apex_base + dx_calibrated_mm * 1e-3

        # Local window used ONLY to build a meaningful amplitude baseline for the reliability mask.
        sx_lo = max(0, np.searchsorted(ff_x_traces, x_pred - 0.75 * lam))
        sx_hi = min(len(ff_x_traces), np.searchsorted(ff_x_traces, x_pred + 0.75 * lam))
        rel, reliable = clssa_profile_with_mask(base_img, mon_img, sx_lo, sx_hi,
                                                 ff_dz_mig, iz_peak, **CLSSA_PARAMS)
        x_local = ff_x_traces[sx_lo:sx_hi]

        # Reported value: the trace NEAREST x_pred -- anchored, not a search for the best bin.
        j_pred = int(np.argmin(np.abs(x_local - x_pred)))

        anchored_rows.append(dict(
            method=method_name, scenario=label, dx_calibrated_mm=dx_calibrated_mm,
            x_pred=x_pred, phi0_at_dx_deg=rel[j_pred], amplitude_reliable=bool(reliable[j_pred]),
            true_dx_mm=true_dx_by_scenario[label] * 1e3,
        ))

anchored_df = pd.DataFrame(anchored_rows)
anchored_df[['method', 'scenario', 'dx_calibrated_mm', 'phi0_at_dx_deg', 'amplitude_reliable', 'true_dx_mm']]


**For Kirchhoff/Gazdag at $1\lambda$, $\tfrac12\lambda$, $\tfrac14\lambda$, this
works cleanly: $136°$–$159°$, read with no search at all, exactly at the location the phase-plane
fit's own $\Delta x$ estimate predicts.** This is the pipeline's headline material-change value --
not the diluted global analytic-channel $\phi_0$ from \S4, which this section supersedes for that
purpose -- and it is only as good as the $\Delta x$ estimate that points to it, which is the whole
point of tying the two together this way.

**At $\tfrac18\lambda$, the anchored reading misses**: Kirchhoff gives $-2.8°$, Gazdag $+7.3°$ at
$x_\text{pred}$ itself, even though \S9.3 already showed a genuine $\approx95$–$150°$ feature sits
just **one grid trace ($10\,\text{mm}$) away**. The true displacement at this scale ($14\,\text{mm}$)
is itself smaller than the migration grid spacing, so a single-trace-anchored reading is exactly as
precise as that grid allows -- it can land one cell short of the real feature. This is reported
honestly rather than smoothed over: tying $\phi_0$ to $\Delta x$ this tightly inherits $\Delta x$'s
own resolution limit.

**For Back-propagation, the anchored reading is erratic and sign-inconsistent** ($+79°, -36°,
-35°, +4°$ across the four scales) -- no trace of the $\approx140$–$160°$ plateau Kirchhoff/Gazdag
show. **Note what the `amplitude_reliable` column does *not* catch here: it flags every
Back-propagation row as reliable.** The amplitude-null mask only catches one specific failure mode
-- a genuine envelope zero-crossing -- and Back-propagation's problem (per \S4.4) is broader phase
incoherence across its whole band, which a single trace's amplitude says nothing about. **This
column is a necessary check, not a sufficient one**: trusting an anchored $\phi_0$ reading still
requires the kind of band-wide coherence diagnostic \S4.4 already ran, not just a local amplitude
sanity check.

### 10.1 Educational Aside: Why the $\tfrac18\lambda$ Reading Misses, and Fixing It by Direction-Aware Trace Selection

This subsection exists purely to make the $\tfrac18\lambda$ miss above mechanically visible and to
show it can be fixed -- **not** to propose a replacement for \S10's anchored reading. Dumping every
trace around $x_\text{pred}$ for Kirchhoff at $\tfrac18\lambda$, with no skipping, shows exactly
what happened:

| $x$ [m] | dist. from $x_\text{pred}$ | amplitude | rel\_phi |
|---|---|---|---|
| 1.690 | $-14$mm | normal | $-6.8°$ |
| 1.700 | $-4$mm | normal | $-2.8°$ ← *nearest trace to $x_\text{pred}$, \S10 reports this* |
| 1.710 | $+6$mm | **collapsed** ($0.93$ vs. typical $4.0$) | $+111.6°$ (unreliable) |
| 1.720 | $+16$mm | normal | $+149.1°$ ← genuine signal |
| 1.730 | $+26$mm | normal | $+2.9°$ |

$x_\text{pred}$ sits almost exactly on the boundary between the unchanged-background ramp and the
genuine spike, and the nearest grid trace happens to fall $2\text{--}4\,\text{mm}$ on the *wrong*
side of that boundary -- the immediate next trace in the direction the front moved is a genuine
amplitude null (correctly flagged unreliable), and the real signal is one trace further still.

In [ ]:
def anchored_directional(base_img, mon_img, x_apex_base, dx_calibrated_mm, dz, iz,
                          n_forward=3, half_width_lam=0.75, **clssa_params):
    """Educational variant of Sec 10's anchored reading: starting from the trace nearest
    x_pred, step up to n_forward cells in the direction the front moved (sign of the
    already-trusted calibrated dx), skip amplitude-unreliable cells, and take whichever
    RELIABLE candidate (nearest, or one of the forward steps) has the larger |rel_phi|.
    Uses the sign of an estimate this notebook already trusts -- it does not discover
    displacement direction independently."""
    x_pred = x_apex_base + dx_calibrated_mm * 1e-3
    sx_lo = max(0, np.searchsorted(ff_x_traces, x_pred - half_width_lam * lam))
    sx_hi = min(len(ff_x_traces), np.searchsorted(ff_x_traces, x_pred + half_width_lam * lam))
    rel, reliable = clssa_profile_with_mask(base_img, mon_img, sx_lo, sx_hi, dz, iz, **clssa_params)
    x_local = ff_x_traces[sx_lo:sx_hi]

    j_nearest = int(np.argmin(np.abs(x_local - x_pred)))
    step = 1 if dx_calibrated_mm >= 0 else -1
    candidates = [j_nearest] + [j_nearest + s * step for s in range(1, n_forward + 1)
                                 if 0 <= j_nearest + s * step < len(x_local)]
    reliable_candidates = [j for j in candidates if reliable[j]]
    chosen = max(reliable_candidates, key=lambda j: abs(rel[j])) if reliable_candidates else j_nearest
    return x_local[j_nearest], x_local[chosen], rel[j_nearest], rel[chosen]


directional_rows = []
for method_name, base_stack in MIGRATION_METHODS.items():
    x_apex_base = locate_baseline_front(base_stack)
    base_img = np.nan_to_num(base_stack[0])
    env_base = np.abs(sp_hilbert(base_img, axis=0))
    iz_peak  = int(np.argmax(env_base[:, np.searchsorted(ff_x_traces, x_apex_base)]))

    for label in SCENARIOS_FOR_PIPELINE:
        i_scen  = ff_scenarios.index(label)
        mon_img = np.nan_to_num(base_stack[i_scen])
        dx_calibrated_mm = float(results_df.loc[
            (results_df['method'] == method_name) & (results_df['scenario'] == label),
            'dx_calibrated_mm'].iloc[0])

        x_near, x_chosen, phi_near, phi_chosen = anchored_directional(
            base_img, mon_img, x_apex_base, dx_calibrated_mm, ff_dz_mig, iz_peak, **CLSSA_PARAMS)
        directional_rows.append(dict(method=method_name, scenario=label,
                                      phi0_nearest_deg=phi_near, phi0_directional_deg=phi_chosen,
                                      moved=(x_chosen != x_near)))

directional_df = pd.DataFrame(directional_rows)
directional_df


**The $\tfrac18\lambda$ miss is fixed for Kirchhoff and Gazdag** ($-2.8°\to
+149.1°$ and $+7.3°\to+95.8°$), and **the scales that already worked are left alone**
($\tfrac14\lambda$, $\tfrac12\lambda$ unchanged; $1\lambda$ shifts by only a few degrees within
the same already-correct $\approx145°$ plateau) -- the directional step does not introduce
spurious changes where the simple nearest-trace reading was already adequate.

**Back-propagation is the negative control that matters here.** Its $\tfrac18\lambda$ and
$\tfrac14\lambda$ readings do change (the directional step finds different nearby traces), but
land on *different small, erratic* values ($+4.0°\to-24.6°$, $-34.7°\to-36.1°$), not on anything
resembling Kirchhoff/Gazdag's $\approx100$–$150°$ plateau. That is exactly what should happen if
this fix addresses a **grid-resolution targeting problem** specific to a narrow, well-localised
spike (Kirchhoff/Gazdag) rather than Back-propagation's separate, genuine phase-incoherence
problem (\S4.4) -- there is no nearby reliable large signal for the directional step to find,
because the underlying problem is not "the answer is one cell over," it is "there reliably is no
clean answer nearby." **This subsection is a demonstration of the resolution mechanism, not a
recommended replacement for \S10's anchored reading** -- the number of forward steps searched and
the choice to prefer the larger-magnitude reliable candidate are choices tuned to this dataset's
$10\,\text{mm}$ grid, not a generally-validated rule for unknown sub-cell displacements.

### 10.2 Pushing Further: $\tfrac{1}{16}\lambda$ and $\tfrac{1}{32}\lambda$ -- the Directional Fix Finds the Right Trace, but the Signal Itself Is Fading

`migrated_results.npz` has two scenarios smaller than anything used so far ($\tfrac{1}{16}\lambda
\approx7\,\text{mm}$, $\tfrac{1}{32}\lambda\approx4\,\text{mm}$ true displacement -- both *below*
half the $10\,\text{mm}$ migration grid spacing). Running \S10.1's directional fix on them tests
whether the $\tfrac18\lambda$ recovery was a one-off or generalises -- and, more importantly,
whether "find the right trace" is the *only* thing standing between this method and a full
recovery at arbitrarily small displacement.

In [ ]:
SMALLEST_SCENARIOS = ['⅛λ', '¹⁄₁₆λ', '¹⁄₃₂λ']

smallest_rows = []
for method_name, base_stack in MIGRATION_METHODS.items():
    x_apex_base = locate_baseline_front(base_stack)
    base_img = np.nan_to_num(base_stack[0])
    env_base = np.abs(sp_hilbert(base_img, axis=0))
    iz_peak  = int(np.argmax(env_base[:, np.searchsorted(ff_x_traces, x_apex_base)]))

    for label in SMALLEST_SCENARIOS:
        i_scen  = ff_scenarios.index(label)
        mon_img = np.nan_to_num(base_stack[i_scen])
        # these two scales were never in results_df -- apply the existing Sec 4.2 factor directly
        ix_lo, ix_hi = crop_indices(x_apex_base, half_width_lam=2.5)
        base_crop, mon_crop = base_img[:, ix_lo:ix_hi], mon_img[:, ix_lo:ix_hi]
        _, dx_r, _, *_ = estimate_shift_2d(base_crop, mon_crop, ff_dz_mig, ff_dx_mig, ff_kz_c)
        dx_calibrated_mm = dx_r * 1e3 * calibration_factors[method_name]

        x_near, x_chosen, phi_near, phi_chosen = anchored_directional(
            base_img, mon_img, x_apex_base, dx_calibrated_mm, ff_dz_mig, iz_peak, **CLSSA_PARAMS)
        smallest_rows.append(dict(method=method_name, scenario=label,
                                   true_dx_mm=true_dx_by_scenario[label] * 1e3,
                                   dx_calibrated_mm=dx_calibrated_mm,
                                   phi0_nearest_deg=phi_near, phi0_directional_deg=phi_chosen))

smallest_df = pd.DataFrame(smallest_rows)
smallest_df


**The directional fix keeps working as a targeting tool -- it is not the
limiting factor any more.** But the *magnitude it finds* decays steadily as true displacement
shrinks below the grid spacing:

| Method | $\tfrac18\lambda$ (14mm) | $\tfrac{1}{16}\lambda$ (7mm) | $\tfrac{1}{32}\lambda$ (4mm) |
|---|---|---|---|
| Gazdag | $96°$ | $69°$ | $47°$ |
| Kirchhoff | $149°$ | $9°$ | $3°$ |

Gazdag degrades gradually; **Kirchhoff collapses almost completely** between $\tfrac18\lambda$ and
$\tfrac{1}{16}\lambda$ -- a real, method-dependent asymmetry at this extreme end, not noise (both
numbers come from the same directional-search procedure that worked cleanly for both methods at
every larger scale).

**This is a different, deeper limit than \S10.1's targeting problem.** \S10.1 showed the
$\tfrac18\lambda$ miss was fixable because the *full-sized* $\approx150°$ signal genuinely existed
one cell away -- only its *location* was off. Here, even with the directional search correctly
finding the best nearby candidate every time, the signal *itself* is smaller, because the
underlying physical perturbation (a few mm of front advance on a $10\,\text{mm}$ grid) produces a
proportionally weaker diffraction response than a full grid-cell displacement does, in a way that
no choice of which trace to read can recover. No amount of clever trace selection fixes a signal
that is genuinely fading at its source -- this is the resolution floor itself, not an artefact of
where the analysis happened to look.

## 11. Kinematic-to-Dynamic Decomposition: Shift-and-Subtract Before Reading Material Change

Every estimator so far has asked one fit (or one CLSSA reading) to account for displacement and
material change at the same time, at the same location. A cleaner alternative: separate the
**kinematic** problem (where did the front move to?) from the **dynamic** problem (how did the
medium change?) into two explicit steps, so the second step never has to fight the first.

1. **Kinematic tracking.** Estimate $\Delta x$ from the real/Hermitian channel alone -- this
   notebook already does this in \S4.2, with the intercept fixed at $0$ by Hermitian symmetry
   rather than fitted, which is the cleanest version of "don't let the plane fit both quantities
   at once."
2. **Shift-and-subtract.** Physically shift the baseline image by the estimated $\Delta x$ using
   the Fourier shift theorem ($\widehat{B}(\kappa_x)\,e^{-i\kappa_x\Delta x}$), so the baseline and
   monitor are now spatially aligned. Any quantity computed from the aligned pair no longer has to
   share a design matrix with a translation term -- the translation has been removed from the
   *data*, not just modelled alongside the rest.
3. **Dynamic estimation.** Re-read the material-change signal -- here, \S9/\S10's CLSSA-at-the-
   predicted-location diagnostic -- on the aligned pair. If a real, independent material change
   exists, it should survive this step intact; if the original large reading was actually an
   uncompensated-translation artefact, it should not.

That third point is the one this section tests directly, with a control case designed specifically
to catch the failure mode of "the alignment step quietly cancels real signal along with the
artefact" before trusting the real-data result.

In [ ]:
def fourier_shift_x(img, dx_g, shift_m):
    """Sub-pixel shift of a 2D (z,x) image along x by shift_m metres, via the Fourier shift
    theorem -- the "shift" step of shift-and-subtract."""
    Nz, Nx = img.shape
    kx_ax = np.fft.fftfreq(Nx, d=dx_g) * 2 * np.pi
    phase = np.exp(-1j * kx_ax * shift_m)
    return np.real(np.fft.ifft(np.fft.fft(img, axis=1) * phase[None, :], axis=1))


### 11.1 Control Test: Does Alignment Destroy a Genuine Material-Change Signal?

\S8 already validated a clean, single-reflector conductivity-driven material change with *zero*
displacement (phase rotation matches Fresnel theory exactly). That case cannot catch the failure
mode of interest here, because with zero true displacement there is nothing for the shift step to
remove in the first place. The test that matters is **both effects present at once**: a real
$0.3\lambda$ displacement *and* a real $+30\%$ conductivity change (the exact case already
validated against Fresnel theory in \S8.4, $\phi_0\approx+0.45°$), applied together to the same
synthetic reflector.

In [ ]:
dx_true_demo = 0.30 * lam
gauss_x_at = lambda center: np.exp(-0.5 * ((x_conv - center) / (lam / 2)) ** 2)
x_center_demo = x_conv.mean()

trace_base_demo = dispersive_reflectivity(6.0, 5.0)
trace_mon_demo  = dispersive_reflectivity(6.0, 5.0 * 1.30)   # +30% conductivity, Sec 8.4's validated case

base_full_demo = np.outer(trace_base_demo, gauss_x_at(x_center_demo))
mon_full_demo  = np.outer(trace_mon_demo,  gauss_x_at(x_center_demo + dx_true_demo))
base_crop_demo, mon_crop_demo = base_full_demo[iz_lo_conv:iz_hi_conv, :], mon_full_demo[iz_lo_conv:iz_hi_conv, :]

# Step 1: kinematic estimate (real channel)
_, dx_r_demo, _, _, _, _ = estimate_shift_2d(base_crop_demo, mon_crop_demo, dz_conv, dx_conv, kz_c)
print(f"True dx = {dx_true_demo*1e3:.3f} mm   |   Step 1 estimated dx_raw = {dx_r_demo*1e3:.3f} mm")

# Step 2: shift the baseline to align with the monitor
base_aligned_demo = fourier_shift_x(base_crop_demo, dx_conv, dx_r_demo)

# Step 3: dynamic (analytic-channel) reading, UNALIGNED vs ALIGNED
a_base_demo, a_mon_demo = sp_hilbert(base_crop_demo, axis=0), sp_hilbert(mon_crop_demo, axis=0)
_, dxres_unaligned, phi0_unaligned, *_ = estimate_shift_2d(
    a_base_demo, a_mon_demo, dz_conv, dx_conv, kz_c, kz_pos_only=True, force_dz_zero=True)

a_base_aligned_demo = sp_hilbert(base_aligned_demo, axis=0)
_, dxres_aligned, phi0_aligned, *_ = estimate_shift_2d(
    a_base_aligned_demo, a_mon_demo, dz_conv, dx_conv, kz_c, kz_pos_only=True, force_dz_zero=True)

print(f"\nUNALIGNED: residual dx = {dxres_unaligned*1e3:+7.3f} mm   phi_0 = {np.degrees(phi0_unaligned):+.3f} deg")
print(f"ALIGNED:   residual dx = {dxres_aligned*1e3:+7.3f} mm   phi_0 = {np.degrees(phi0_aligned):+.3f} deg")


**The kinematic residual is removed completely (the true $33.6\,\text{mm}$
collapses to $0.000\,\text{mm}$) and the material-change reading is preserved to three decimal
places ($+0.447°$ before and after).** Shift-and-subtract does exactly what it claims: it removes
the translation from the data without touching an independent dynamic signal that happens to be
present at the same time. This is the necessary control before trusting what \S11.2 finds when the
same procedure is applied to the real dataset.

### 11.2 Applying Shift-and-Subtract to the Real Data -- an Important Correction

With the method validated, repeat \S10/\S10.1's anchored CLSSA reading -- but on the
shift-and-subtract-aligned baseline, instead of the raw one.

In [ ]:
print(f"{'method':>10s} {'scen':>5s} {'dx_calib[mm]':>13s} {'phi0 UNALIGNED':>15s} {'phi0 ALIGNED':>13s}")

for method_name, base_stack in MIGRATION_METHODS.items():
    x_apex_base = locate_baseline_front(base_stack)
    base_img_full = np.nan_to_num(base_stack[0])
    env_base = np.abs(sp_hilbert(base_img_full, axis=0))
    iz_peak  = int(np.argmax(env_base[:, np.searchsorted(ff_x_traces, x_apex_base)]))

    for label in SCENARIOS_FOR_PIPELINE:
        i_scen = ff_scenarios.index(label)
        mon_img_full = np.nan_to_num(base_stack[i_scen])
        dx_calibrated_mm = float(results_df.loc[
            (results_df['method'] == method_name) & (results_df['scenario'] == label),
            'dx_calibrated_mm'].iloc[0])

        _, _, _, phi_unaligned = anchored_directional(
            base_img_full, mon_img_full, x_apex_base, dx_calibrated_mm, ff_dz_mig, iz_peak, **CLSSA_PARAMS)

        base_aligned_full = fourier_shift_x(base_img_full, ff_dx_mig, dx_calibrated_mm * 1e-3)
        _, _, _, phi_aligned = anchored_directional(
            base_aligned_full, mon_img_full, x_apex_base, dx_calibrated_mm, ff_dz_mig, iz_peak, **CLSSA_PARAMS)

        print(f"{method_name:>10s} {label:>5s} {dx_calibrated_mm:13.2f} {phi_unaligned:15.2f} {phi_aligned:13.2f}")


**For Kirchhoff and Gazdag, the large $\approx96$–$159°$ readings from \S9.3 and
\S10 collapse to $-3.8°$ to $+0.4°$ once the baseline is properly aligned first.** Given \S11.1's
control case just showed this same procedure leaves a genuine material-change signal untouched,
this is not the alignment "washing out" a real signal -- it is removing an artefact that was never
a real, independent material-change signature in the first place.

**This requires an honest correction to \S9 and \S10's interpretation.** Those sections were
right about the *mechanics* -- CLSSA genuinely recovers a large, reproducible, well-localised phase
difference at the predicted front position. What this section shows is that, for *this* dataset,
that large reading is almost entirely an uncompensated-translation artefact: `FluidFlow_Playground.ipynb`
builds every scenario by **rigidly translating** the whole nine-box wetting assembly (\S1.1, \S7),
so there is no independent material-identity change anywhere in the ground truth to find. Reading
CLSSA's phase difference at a fixed location *before* compensating for the fact that the whole
pattern slid past that location is precisely the "extrapolation lever" effect: a real, large
translation-induced phase difference gets read as if it were a material signature, simply because
the comparison was never told the front had moved. \S1.1's framing -- $\phi_0$ as a clean,
decoupled material-change channel -- and \S8's convolution-model validation of that theory remain
correct; what was missing was this section's step of removing the known kinematic component
*before* asking what, if anything, is left over. For this dataset, once that is done properly,
the honest answer is: nothing significant is left over, because nothing in the ground truth
ever changed materially -- only geometrically.

### 11.3 Back-Propagation: the One Case That Doesn't Collapse, and Why That's Informative

Back-propagation's $1\lambda$ scenario is the exception -- its aligned reading stays large rather
than collapsing like every other Kirchhoff/Gazdag case. \S4.2 already flagged exactly why: the
per-scenario calibration factor for Back-propagation at $1\lambda$ ($k\approx1.84$) is an outlier
against the $\approx0.98$–$0.99$ found at every other scale, so the *median* $k$ used throughout
this notebook gives a poor $\Delta x$ estimate specifically there (\S4.2: $60.9\,\text{mm}$
inferred vs. $113\,\text{mm}$ true). Shift-and-subtract can only align the data as well as its own
$\Delta x$ estimate allows -- align by the wrong amount, and a large apparent "dynamic" residual
remains, not because of real material change, but because the kinematic step itself failed.

This is not a weakness specific to this section -- it is the same self-diagnostic property \S10
already relied on (a large predicted-vs-found offset flags an untrustworthy result without needing
ground truth), now appearing as "alignment did not collapse the signal" instead of "the search
landed far from the prediction." Both symptoms point at the same underlying cause: trust
Kirchhoff/Gazdag's $\Delta x$ and the now-validated near-zero dynamic residual that follows from
it; treat Back-propagation's $1\lambda$ result, specifically, as a case where the kinematic
estimate itself should be fixed before any dynamic reading built on top of it is meaningful.

## 12. A True Thin-Layer Model: Resolving a Continuously-Graded Material Change

\S8 modelled material change as a single Fresnel interface (host $\to$ target) and found a clean,
important limit: for two *real* permittivities, $\Gamma$ is itself real, so its phase is pinned at
exactly $0°$ or $180°$ -- never graded. But \cref{sec:th-material-change} (`03_theory.tex`,
"Why a sub-wavelength fracture produces a frequency-independent phase shift") does not model a
sub-wavelength fracture as a single interface -- it is explicitly **two** reflections, at the top
and bottom of the fracture, that overlap because $d\ll\lambda$:
$$R_{\mathrm{total}} \approx R_{\mathrm{top}} + R_{\mathrm{bottom}}\,e^{-j2kd}.$$
At normal incidence $R_{\mathrm{bottom}}=-R_{\mathrm{top}}$ exactly, so the two leading-order terms
**cancel** -- what survives is the interference factor $e^{-j2kd}$ itself, which is inherently
complex regardless of whether the fill is lossy. This is qualitatively different physics from
\S8's single interface, and it is the model the thesis actually intends for a fluid-filled
fracture. This section implements it exactly (not the small-$kd$ approximation used to derive the
thesis's two follow-on formulas) and sweeps the fill **continuously** from air to water, using the
same graded steps `FluidFlow_Playground.ipynb`'s gprMax model actually uses, to test directly
whether this geometry resolves a uniform material change -- and which of the thesis's two
candidate phase formulas the result actually matches.

In [ ]:
eps_m_thin = eps_r_host  # 3.15, same host (ice) as Sec 8
c0_ns = 0.299792458       # m/ns

def thinlayer_reflectivity(eps_f, sigma_f_mS=0.0):
    """Exact two-reflection thin-layer model, R_total(omega) = R_top + R_bottom*exp(-j*2*k_fill*d)
    -- thesis eq. (sec:th-material-change, "thinlayer-1"), evaluated exactly (no kd<<1 expansion).
    Reuses Sec 8's existing Ricker spectrum (W_z_conv) and frequency axis (f_axis_GHz)."""
    eps_f_c = eps_complex(eps_f, sigma_f_mS, f_axis_GHz)
    eps_m_c = eps_m_thin + 0j
    R_top = fresnel_gamma(eps_m_c, eps_f_c)
    R_bot = fresnel_gamma(eps_f_c, eps_m_c)
    omega = 2 * np.pi * f_axis_GHz * 1e9
    k_fill = omega * np.sqrt(eps_f_c) / (c0_ns * 1e9)
    R_total_f = R_top + R_bot * np.exp(-1j * 2 * k_fill * d_frac)
    return np.real(np.fft.ifft(W_z_conv * R_total_f))

d_frac = lam / 20   # fracture thickness, sub-wavelength (matches PhasePlaneFit.ipynb's Material Change Test)


### 12.1 Sweeping the Fill Uniformly From Air to Water

Use the *exact* graded steps already present in the real dataset's gprMax materials (ice host,
air, `wet_zone_10..70`, water -- \S1.1's caveat about this fixed graded ramp), so the result is
directly comparable to the rest of this notebook, not an arbitrary new sweep.

In [ ]:
GRADED_STEPS = [(1.0, 0.0), (10.0, 0.00125), (20.0, 0.0025), (30.0, 0.00375), (40.0, 0.005),
                (50.0, 0.00625), (60.0, 0.0075), (70.0, 0.00875), (80.0, 0.01)]  # (eps_r, sigma S/m)

trace_base_thin = thinlayer_reflectivity(1.0, 0.0)   # baseline fill = air
base_full_thin  = np.outer(trace_base_thin, gauss_x)
base_crop_thin  = base_full_thin[iz_lo_conv:iz_hi_conv, :]
a_base_thin     = sp_hilbert(base_crop_thin, axis=0)

thinlayer_rows = []
for eps_f, sigma_S in GRADED_STEPS:
    sigma_mS = sigma_S * 1e3
    trace_mon_thin = thinlayer_reflectivity(eps_f, sigma_mS)
    mon_crop_thin = np.outer(trace_mon_thin, gauss_x)[iz_lo_conv:iz_hi_conv, :]
    a_mon_thin = sp_hilbert(mon_crop_thin, axis=0)
    _, _, phi0_fit, *_ = estimate_shift_2d(
        a_base_thin, a_mon_thin, dz_conv, dx_conv, kz_c, kz_pos_only=True, force_dz_zero=True)
    label = 'air' if eps_f == 1 else ('water' if eps_f == 80 else f'wet_{int(eps_f)}')
    thinlayer_rows.append(dict(fill=label, eps_r=eps_f, sigma_mS=sigma_mS,
                                phi0_fitted_deg=np.degrees(phi0_fit)))

thinlayer_df = pd.DataFrame(thinlayer_rows)
thinlayer_df


**The phase channel resolves the graded fill smoothly and monotonically**: $0°$
(air, by definition) $\to-166°\to-158°\to\dots\to-130°$ (water), strictly decreasing in magnitude
at every step. Unlike \S8's single interface, this is a genuinely **continuous, in-principle
invertible** signature -- exactly what \cref{sec:th-material-change} claims a sub-wavelength
fracture should produce, and exactly what \S8 showed a single Fresnel interface cannot.

### 12.2 Which Theoretical Formula Actually Matches?

\cref{sec:th-material-change} gives two candidate expressions for this phase: the exact ratio of
two `thinlayer-1` reflectivities (eq. "thinlayer-ratio"/"thinlayer-phase"), and a separate
closed-form approximation, eq. "intercept-material",
$$c = \Delta\theta \approx \frac{2\pi d}{\lambda}\left(\sqrt{\varepsilon_{\mathrm{fluid}}} -
\sqrt{\varepsilon_{\mathrm{baseline}}}\right).$$
These are different expressions, and for an air$\to$water contrast this large there is no a priori
guarantee they agree -- check both against the actual fitted result.

In [ ]:
def theory_ratio_dtheta(eps_f_base, eps_f_mon, sigma_base_mS=0.0, sigma_mon_mS=0.0):
    """Exact thinlayer-1 ratio, R_mon/R_base, evaluated at f_c with complex Fresnel coefficients
    -- no small-kd expansion, matching the sign convention used throughout this notebook."""
    eps_m_c = eps_m_thin + 0j
    def _R_total(eps_f, sigma_mS):
        eps_f_c = eps_complex(eps_f, sigma_mS, f_c)
        Rt, Rb = fresnel_gamma(eps_m_c, eps_f_c), fresnel_gamma(eps_f_c, eps_m_c)
        k_fill = 2 * np.pi * f_c * 1e9 * np.sqrt(eps_f_c) / (c0_ns * 1e9)
        return Rt + Rb * np.exp(-1j * 2 * k_fill * d_frac)
    R_base, R_mon = _R_total(eps_f_base, sigma_base_mS), _R_total(eps_f_mon, sigma_mon_mS)
    return -np.degrees(np.angle(R_mon / R_base))


def theory_intercept_formula(eps_f_base, eps_f_mon):
    """eq. intercept-material, literally as written in the thesis."""
    return np.degrees((2 * np.pi * d_frac / lam) * (np.sqrt(eps_f_mon) - np.sqrt(eps_f_base)))


thinlayer_df['theory_ratio_deg'] = [
    theory_ratio_dtheta(1.0, row.eps_r, 0.0, row.sigma_mS) for row in thinlayer_df.itertuples()
]
thinlayer_df['theory_intercept_deg'] = [
    theory_intercept_formula(1.0, row.eps_r) for row in thinlayer_df.itertuples()
]
thinlayer_df[['fill', 'eps_r', 'phi0_fitted_deg', 'theory_ratio_deg', 'theory_intercept_deg']]


**The fitted result tracks the exact ratio formula closely in sign, magnitude,
and trend** ($-166°$ fitted vs. $-166°$ ratio-theory at the first step; both decreasing in
magnitude toward water, agreement within $\approx30°$ at the water end, where the residual gap is
explained the same way \S8.3 explained its own theory-vs-fit gap: a single-frequency evaluation at
$f_c$ does not equal a band-weighted fit's effective frequency, and water's much slower wave speed
pushes the fracture's electrical thickness $kd$ furthest from the small-$kd$ regime where the
approximation is cleanest).

**eq. "intercept-material" is qualitatively wrong for this contrast** -- wrong sign throughout, and
*increasing* in magnitude toward water where the fitted and exact-ratio results both *decrease*.
That formula was derived as a small-signal linearisation; $\sqrt{\varepsilon_{\mathrm{water}}}-
\sqrt{\varepsilon_{\mathrm{air}}}=9-1=8$ is not a small perturbation, and the linearisation does not
survive contact with the exact physics at this scale. **This is a concrete correction worth making
to the thesis text**: eq. "intercept-material" should either be restricted to genuinely small
permittivity contrasts, or replaced with the exact ratio (eq. "thinlayer-ratio"/"thinlayer-phase",
evaluated with complex Fresnel coefficients as done here) for anything resembling the air/water
contrast actually used in \cref{ch:fluidflow}.

### 12.3 Conclusion: Yes -- a Genuine Fracture's Material Change Is Resolvable

This directly answers the question \S8 left open. A single Fresnel interface cannot see a pure
permittivity change in its phase (\S8's central finding); a genuine two-reflection sub-wavelength
fracture **can**, smoothly and monotonically, with no conductivity required at all -- because the
thin layer's near-cancelling top and bottom reflections make the surviving response inherently
complex, a property of the *geometry* (two reflections interfering), not of material loss. This
confirms \cref{sec:th-material-change}'s qualitative claim that a sub-wavelength fracture produces
a resolvable, frequency-independent phase shift, while showing the specific closed-form formula
given for its size (eq. "intercept-material") needs correcting -- the exact ratio is what the
physics, and this notebook's own validated estimator, actually agree on.